# Job Preprocessing Pipeline v3 — PhoBERT + Cosine Similarity

Mục tiêu:
- Chuẩn hoá dữ liệu job posting
- Trích xuất metadata + skills
- Tạo text tối ưu cho PhoBERT
- Sinh embedding cho matching / retrieval
- Dùng cosine similarity cho semantic search
- Xuất artifact cho matching, chatbot, section retrieval

In [79]:
import re
import os
import json
import math
import html
import time
import unicodedata
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Optional: để word segmentation tiếng Việt
try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except Exception:
    HAS_UNDERTHESEA = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

NOTEBOOK_VERSION = "preprocessing_v3_phobert"
# RAW_INPUT_PATH = Path("../data/raw/jobs/topcv_all_fields_merged_2026-03-21_19-27-47")   # ← sửa
RAW_INPUT_PATH = Path("../../../data/raw/jobs/topcv_all_fields_merged_2026-04-01_19-41-42.csv")
ARTIFACT_DIR = Path("../outputs_preprocessing_v3_test")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# RUN_EMBEDDING = True
# RUN_SECTION_EMBEDDING = True

RUN_EMBEDDING = False
RUN_SECTION_EMBEDDING = False

SAVE_INTERMEDIATE = False

PHOBERT_MODEL_NAME = "vinai/phobert-base"
PHOBERT_MAX_LENGTH_MATCH = 256
PHOBERT_MAX_LENGTH_CHATBOT = 320
PHOBERT_MAX_LENGTH_CHUNK = 256
PHOBERT_BATCH_SIZE = 16
NORMALIZE_EMBEDDINGS = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("NOTEBOOK_VERSION:", NOTEBOOK_VERSION)
print("RAW_INPUT_PATH:", RAW_INPUT_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())
print("RUN_EMBEDDING:", RUN_EMBEDDING)
print("RUN_SECTION_EMBEDDING:", RUN_SECTION_EMBEDDING)
print("PHOBERT_MODEL_NAME:", PHOBERT_MODEL_NAME)
print("DEVICE:", DEVICE)
print("HAS_UNDERTHESEA:", HAS_UNDERTHESEA)

NOTEBOOK_VERSION: preprocessing_v3_phobert
RAW_INPUT_PATH: ..\..\..\data\raw\jobs\topcv_all_fields_merged_2026-04-01_19-41-42.csv
ARTIFACT_DIR: D:\TTTN\job_recommendation_system\src\data_processing\outputs_preprocessing_v3_test
RUN_EMBEDDING: False
RUN_SECTION_EMBEDDING: False
PHOBERT_MODEL_NAME: vinai/phobert-base
DEVICE: cpu
HAS_UNDERTHESEA: True


In [80]:
# Chuẩn hóa giá trị rỗng
def normalize_empty_value(val):
    if val is None:
        return None
    if isinstance(val, float) and pd.isna(val):
        return None

    s = str(val).strip()
    if not s:
        return None

    lowered = s.lower().strip()
    empty_tokens = {
        "nan", "none", "null", "n/a", "na", "-", "--", "---",
        "không", "không rõ", "chưa rõ", "chưa cập nhật", "đang cập nhật",
        "not specified", "unknown"
    }
    return None if lowered in empty_tokens else s

# Chuẩn hóa thành chuỗi
def safe_str(x):
    x = normalize_empty_value(x)
    return "" if x is None else str(x)

# Lấy Series từ DataFrame, nếu không có thì trả về Series mặc định
def get_series(df, col, default=None):
    if col in df.columns:
        return df[col]
    if default is None:
        default = [None] * len(df)
    return pd.Series(default, index=df.index)

# Lấy giá trị đầu tiên không rỗng từ danh sách các giá trị
def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

# Chuẩn hóa unicode
def normalize_unicode(text):
    text = safe_str(text)
    return unicodedata.normalize("NFC", text)

# Xóa thẻ html
def remove_html(text):
    text = safe_str(text)
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

# Chuẩn hóa các kiểu gạch đầu dòng khác nhau về "-"
def normalize_dash(text):
    text = safe_str(text)
    dash_map = {
        "–": "-",
        "—": "-",
        "−": "-",
        "•": "-",
        "●": "-",
        "▪": "-",
        "►": "-",
        "✅": "-",
        "✔": "-",
    }
    for k, v in dash_map.items():
        text = text.replace(k, v)
    return text

# Loại bỏ phần tử trùng lặp trong list, giữ nguyên thứ tự
def deduplicate_list(values):
    out = []
    seen = set()
    for v in values:
        key = safe_str(v).strip()
        if not key:
            continue
        if key not in seen:
            seen.add(key)
            out.append(v)
    return out
# Chuẩn hóa văn bản để phục vụ cho việc matching, loại bỏ dấu câu không cần thiết, chuyển về chữ thường, chuẩn hóa unicode, loại bỏ dấu tiếng Việt để tăng khả năng matching
def normalize_for_match(text):
    text = clean_text_light(text)
    if not text:
        return ""

    text = text.lower()
    text = text.replace("đ", "d")
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^\w\s/\+\-#\.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Loại bỏ dòng trùng lặp trong văn bản, giữ nguyên thứ tự và cấu trúc cơ bản
def deduplicate_text_lines(text, min_key_len=12):
    text = safe_str(text)
    if not text:
        return ""
    out_lines = []
    seen = set()
    for line in text.splitlines():
        raw = line.strip()
        if not raw:
            continue
        key = re.sub(r"\s+", " ", raw.lower())
        if len(key) < min_key_len:
            out_lines.append(raw)
            continue
        if key not in seen:
            seen.add(key)
            out_lines.append(raw)
    return "\n".join(out_lines).strip()

# Các hàm làm sạch văn bản ở nhiều mức độ khác nhau
# Làm sạch nhẹ, giữ nguyên cấu trúc cơ bản (giữ format gốc, chỉ loại bỏ những thứ thực sự không cần thiết)
def clean_text_light(text):
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)
    text = text.replace("\\n", "\n")
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = deduplicate_text_lines(text)
    return text.strip()

# Làm sạch nghiêm ngặt, loại bỏ hầu hết dấu câu, chỉ giữ lại chữ và số (giữ nguyên cấu trúc cơ bản như xuống dòng và gạch đầu dòng để preserve meaning)
def clean_text_preserve_structure(text):
    text = clean_text_light(text)
    if not text:
        return ""
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n\s*-\s*", "\n- ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Làm sạch nghiêm ngặt hơn, chỉ giữ lại chữ, số và một số dấu câu cơ bản (để vector hóa và matching)
def clean_text_strict(text):
    text = clean_text_light(text)
    if not text:
        return ""
    text = text.lower() # chuyển về chữ thường để chuẩn hóa
    text = re.sub(r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ/+\-#\.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Làm sạch cho PhoBERT, giữ lại dấu câu cần thiết để preserve meaning 
def clean_text_for_phobert(text):
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)
    text = text.replace("\\n", "\n")

    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # giữ dấu câu đủ để preserve meaning cho PhoBERT
    text = re.sub(r"[^\w\sÀ-ỹ\.,;:/\-\+\#\(\)%\n]", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)

    return text.strip()

# Rút gọn văn bản theo số lượng từ, giữ nguyên cấu trúc cơ bản
def truncate_by_words(text, max_words=220):
    text = safe_str(text)
    if not text:
        return ""
    words = text.split()
    return " ".join(words[:max_words]).strip()

# Lưu DataFrame thành Parquet nếu có thể, nếu không thì lưu CSV
def save_table(df, base_path: Path):
    base_path = Path(base_path)
    try:
        out_path = str(base_path) + ".parquet"
        df.to_parquet(out_path, index=False)
        return out_path
    except Exception:
        out_path = str(base_path) + ".csv"
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        return out_path

In [81]:
def load_raw_data(input_path: Path) -> pd.DataFrame:
    if not input_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {input_path}")

    suffix = input_path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(input_path)
    elif suffix == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Unsupported file format: {suffix}")

    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df


df_raw = load_raw_data(RAW_INPUT_PATH)
display(df_raw.head(3))
print(df_raw.columns.tolist())

[INFO] Loaded raw data: 1477 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,detail_salary,address_list,detail_location,exp_list,detail_experience,deadline,tags,job_level,education_level,job_quantity,employment_type,working_addresses,working_times,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Automation Tester,1,Nhân Viên Tester Mobile App (Android)- Đi Làm Ngay,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,CÔNG TY CỔ PHẦN ĐẦU TƯ AHV HOLDING,CÔNG TY CỔ PHẦN ĐẦU TƯ AHV HOLDING,https://www.topcv.vn/brand/ahv?id=223331,https://www.topcv.vn/brand/ahv?id=223331,Thoả thuận,Thoả thuận,Hà Nội,Hà Nội,1 năm,1 năm,13/04/2026,Chuyên môn Software Tester (Automation & Manual); IT - Phần cứng và máy tính; IT - Phần mềm; Thương mại điện tử,Nhân viên,Đại Học trở lên,2 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)",Thứ 2 - Thứ 6 (từ 08:30 đến 18:00),"Phân tích yêu cầu sản phẩm và thiết kế để xây dựng test case, test plan cho ứng dụng mobile. Thực hiện kiểm thử ứng dụng trên nhiều thiết bị thực tế và trình giả lập (emulator/simulator). Kiểm thử...","Tốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan. Có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. Có kinh nghiệm về sản phẩm Product android Nắ...","Được sử dụng thiết bị test (smartphone thật, tool nội bộ). BHXH – BHYT – BHTN đầy đủ theo luật. Du lịch, teambuilding, sinh nhật, thưởng lễ tết, đào tạo kỹ năng. Lộ trình phát triển rõ ràng: Cơ hộ...",https://ahvholding.com/,NaN,25-99,NaN,NaN,"Số nhà 18, Tổ 1, Phường Phố Cò, Thành phố Sông Công, Tỉnh Thái Nguyên, Việt Nam","Công ty Cổ phần Đầu tư AHV Holding là một tổ chức đầu tư đa ngành, được thành lập với sứ mệnh kiến tạo hệ sinh thái doanh nghiệp vững mạnh, bền vững và có tính kết nối cao. Với nền tảng tài chính ..."
1,https://www.topcv.vn/brand/cennext/tuyen-dung/wordpress-developer-senior-level-20h-05h-remote-j1867723.html,Frontend Developer,1,WordPress Developer - Senior Level - (20h - 05h) Remote,WordPress Developer - Senior Level - (20h - 05h) Remote,CENNEXT,CENNEXT,https://www.topcv.vn/brand/cennext?id=5338,https://www.topcv.vn/brand/cennext?id=5338,12 - 18 triệu,12 - 18 triệu,Hà Nội & 3 nơi khác,Hà Nội và 3 nơi khác,3 năm,3 năm,05/04/2026,Chuyên môn Frontend Developer; IT - Phần mềm,Nhân viên,Cao Đẳng trở lên,3 người,Toàn thời gian,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,NaN,"Build websites and landing pages on WordPress Customize and extend themes/plugins as required by each project. Publish content: create/edit Pages & Posts, set categories/tags, featured image, slug...",">3 years of hands-on WordPress publishing/site build experience. Good HTML/CSS/JS; PHP for theme/template tweaks. WordPress admin skills: Pages/Posts, Menus, Media, Users, Permalinks. Confident wi...","Competitive salary, negotiable based on capability. 85% salary during probation, with annual salary review. 13th-month salary, holiday bonuses, and Best Member awards. Training and development pro...",https://cennext.com/,NaN,1000+,NaN,NaN,"Tầng 3, Nhà D, số 22 Thành Công, Ba Đình, Hà Nội","Được thành lập vào năm 2013, CENNEXT là một công ty chuyên về Dịch vụ xử lý dữ liệu từ xa trong lĩnh vực Thương mại điện tử (dịch thuật, quản lý website, facebook, chỉnh sửa photoshop...) cho các ..."
2,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Analyst,1,Data Processing Analyst,Data Pr

['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [82]:
# Import dữ liệu vào schema chung
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["job_url"] = get_series(df, "job_url")
    out["job_id"] = get_series(df, "job_id")

    out["job_title_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "detail_title"),
            get_series(df, "title")
        )
    ]
    out["job_slug_raw"] = get_series(df, "source_field_name")

    out["salary_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "detail_salary"),
            get_series(df, "salary_list")
        )
    ]

    out["location_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "detail_location"),
            get_series(df, "address_list")
        )
    ]

    out["working_addresses_raw"] = get_series(df, "working_addresses")

    out["working_times_raw"] = get_series(df, "working_times")

    out["experience_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "exp_list"),
            get_series(df, "detail_experience")
        )
    ]
    out["education_level_raw"] = get_series(df, "education_level")
    out["employment_type_raw"] = get_series(df, "employment_type")
    out["job_level_raw"] = get_series(df, "job_level")
    out["job_quantity_raw"] = get_series(df, "job_quantity")

    out["description_raw"] = get_series(df, "desc_mota")
    out["requirements_raw"] = get_series(df, "desc_yeucau")
    out["benefits_raw"] = get_series(df, "desc_quyenloi")

    out["tags_raw"] = get_series(df, "tags")

    out["deadline_raw"] = get_series(df, "deadline")

    out["company_name_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_name_full"),
            get_series(df, "company_name")
        )
    ]
    out["company_website_raw"] = get_series(df, "company_website")
    out["company_field_raw"] = get_series(df, "company_field_from_job")

    out["company_scale_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_scale_from_job"),
            get_series(df, "company_scale")
        )
    ]
    
    out["company_address_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_address_from_job"),
            get_series(df, "company_address")
        )
    ]
    out["company_description_raw"] = get_series(df, "company_description")

    return out


df = merge_semantic_columns(df_raw)
print("[INFO] After merging:", df.shape)
display(df.head(3))
df.info()

[INFO] After merging: (1477, 24)


,job_url,job_id,job_title_raw,job_slug_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,education_level_raw,employment_type_raw,job_level_raw,job_quantity_raw,description_raw,requirements_raw,benefits_raw,tags_raw,deadline_raw,company_name_raw,company_website_raw,company_field_raw,company_scale_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,None,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,Automation Tester,Thoả thuận,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)",Thứ 2 - Thứ 6 (từ 08:30 đến 18:00),1 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,2 người,"Phân tích yêu cầu sản phẩm và thiết kế để xây dựng test case, test plan cho ứng dụng mobile. Thực hiện kiểm thử ứng dụng trên nhiều thiết bị thực tế và trình giả lập (emulator/simulator). Kiểm thử...","Tốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan. Có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. Có kinh nghiệm về sản phẩm Product android Nắ...","Được sử dụng thiết bị test (smartphone thật, tool nội bộ). BHXH – BHYT – BHTN đầy đủ theo luật. Du lịch, teambuilding, sinh nhật, thưởng lễ tết, đào tạo kỹ năng. Lộ trình phát triển rõ ràng: Cơ hộ...",Chuyên môn Software Tester (Automation & Manual); IT - Phần cứng và máy tính; IT - Phần mềm; Thương mại điện tử,13/04/2026,CÔNG TY CỔ PHẦN ĐẦU TƯ AHV HOLDING,https://ahvholding.com/,NaN,25-99,"Số nhà 18, Tổ 1, Phường Phố Cò, Thành phố Sông Công, Tỉnh Thái Nguyên, Việt Nam","Công ty Cổ phần Đầu tư AHV Holding là một tổ chức đầu tư đa ngành, được thành lập với sứ mệnh kiến tạo hệ sinh thái doanh nghiệp vững mạnh, bền vững và có tính kết nối cao. Với nền tảng tài chính ..."
1,https://www.topcv.vn/brand/cennext/tuyen-dung/wordpress-developer-senior-level-20h-05h-remote-j1867723.html,None,WordPress Developer - Senior Level - (20h - 05h) Remote,Frontend Developer,12 - 18 triệu,Hà Nội và 3 nơi khác,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,NaN,3 năm,Cao Đẳng trở lên,Toàn thời gian,Nhân viên,3 người,"Build websites and landing pages on WordPress Customize and extend themes/plugins as required by each project. Publish content: create/edit Pages & Posts, set categories/tags, featured image, slug...",">3 years of hands-on WordPress publishing/site build experience. Good HTML/CSS/JS; PHP for theme/template tweaks. WordPress admin skills: Pages/Posts, Menus, Media, Users, Permalinks. Confident wi...","Competitive salary, negotiable based on capability. 85% salary during probation, with annual salary review. 13th-month salary, holiday bonuses, and Best Member awards. Training and development pro...",Chuyên môn Frontend Developer; IT - Phần mềm,05/04/2026,CENNEXT,https://cennext.com/,NaN,1000+,"Tầng 3, Nhà D, số 22 Thành Công, Ba Đình, Hà Nội","Được thành lập vào năm 2013, CENNEXT là một công ty chuyên về Dịch vụ xử lý dữ liệu từ xa trong lĩnh vực Thương mại điện tử (dịch thuật, quản lý website, facebook, chỉnh sửa photoshop...) cho các ..."
2,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,None,Data Processing Analyst,Data Analyst,25 - 30 triệu,Hồ Chí Minh (mới),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:",Thứ 2 - Thứ 6 (từ 09:00 đến 18:00),2 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,1 người,"Get to know the role We are looking for a new Analyst for one of our Data Processing Teams. We believe a successful candidate has problem solving and python programming skills, but if you believ

<class 'pandas.DataFrame'>
RangeIndex: 1477 entries, 0 to 1476
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   job_url                  1477 non-null   str   
 1   job_id                   0 non-null      object
 2   job_title_raw            1477 non-null   str   
 3   job_slug_raw             1477 non-null   str   
 4   salary_raw               1477 non-null   str   
 5   location_raw             1477 non-null   str   
 6   working_addresses_raw    1473 non-null   str   
 7   working_times_raw        1350 non-null   str   
 8   experience_raw           1477 non-null   str   
 9   education_level_raw      1474 non-null   str   
 10  employment_type_raw      1474 non-null   str   
 11  job_level_raw            1474 non-null   str   
 12  job_quantity_raw         1474 non-null   str   
 13  description_raw          1474 non-null   str   
 14  requirements_raw         1474 non-null   str   
 15

In [83]:
# Kiểm tra tỷ lệ trùng lặp, tỷ lệ missing, và phân loại ngôn ngữ
def detect_language_type(text: str) -> str:
    text = safe_str(text)
    if not text:
        return "empty"

    has_vi = bool(re.search(r"[ăâêôơưđàáạảãèéẹẻẽìíịỉĩòóọỏõùúụủũỳýỵỷỹ]", text.lower()))
    has_en = bool(re.search(r"[a-z]", text.lower()))

    if has_vi and has_en:
        return "mixed"
    if has_vi:
        return "vi"
    if has_en:
        return "en"
    return "other"


audit_rows = []
combined_text = (
    get_series(df, "job_title_raw", "") .fillna("") + " " +
    get_series(df, "description_raw", "") .fillna("") + " " +
    get_series(df, "requirements_raw", "") .fillna("")
)

lang_type = combined_text.apply(detect_language_type)

audit_rows.append({
    "n_rows": len(df),
    "dup_by_url_ratio": df["job_url"].duplicated().mean() if "job_url" in df.columns else np.nan,
    "has_minimum_content_ratio": (
        (df["job_title_raw"].fillna("").str.len() > 0) &
        (
            (df["description_raw"].fillna("").str.len() > 0) |
            (df["requirements_raw"].fillna("").str.len() > 0)
        )
    ).mean(),
    "mixed_ratio": (lang_type == "mixed").mean(),
    "en_ratio": (lang_type == "en").mean(),
    "vi_ratio": (lang_type == "vi").mean(),
})

audit_df = pd.DataFrame(audit_rows)
missing_df = pd.DataFrame({
    "column": df.columns,
    "missing_ratio": [df[c].isna().mean() for c in df.columns]
}).sort_values("missing_ratio", ascending=False)

display(audit_df)
display(missing_df)

,n_rows,dup_by_url_ratio,has_minimum_content_ratio,mixed_ratio,en_ratio,vi_ratio
0,1477,0.0,0.997969,0.811781,0.188219,0.0


,column,missing_ratio
1,job_id,1.000000
19,company_website_raw,0.150982
20,company_field_raw,0.097495
7,working_times_raw,0.085985
23,company_description_raw,0.031144
16,tags_raw,0.002708
6,working_addresses_raw,0.002708
17,deadline_raw,0.002708
12,job_quantity_raw,0.002031
15,benefits_raw,0.002031


In [84]:
df_clean = df.copy()

for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(normalize_empty_value)

print("df_raw shape :", df.shape)
print("df_clean shape:", df_clean.shape)
display(df_clean.head(3))

df_raw shape : (1477, 24)
df_clean shape: (1477, 24)


,job_url,job_id,job_title_raw,job_slug_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,education_level_raw,employment_type_raw,job_level_raw,job_quantity_raw,description_raw,requirements_raw,benefits_raw,tags_raw,deadline_raw,company_name_raw,company_website_raw,company_field_raw,company_scale_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,None,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,Automation Tester,Thoả thuận,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)",Thứ 2 - Thứ 6 (từ 08:30 đến 18:00),1 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,2 người,"Phân tích yêu cầu sản phẩm và thiết kế để xây dựng test case, test plan cho ứng dụng mobile. Thực hiện kiểm thử ứng dụng trên nhiều thiết bị thực tế và trình giả lập (emulator/simulator). Kiểm thử...","Tốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan. Có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. Có kinh nghiệm về sản phẩm Product android Nắ...","Được sử dụng thiết bị test (smartphone thật, tool nội bộ). BHXH – BHYT – BHTN đầy đủ theo luật. Du lịch, teambuilding, sinh nhật, thưởng lễ tết, đào tạo kỹ năng. Lộ trình phát triển rõ ràng: Cơ hộ...",Chuyên môn Software Tester (Automation & Manual); IT - Phần cứng và máy tính; IT - Phần mềm; Thương mại điện tử,13/04/2026,CÔNG TY CỔ PHẦN ĐẦU TƯ AHV HOLDING,https://ahvholding.com/,NaN,25-99,"Số nhà 18, Tổ 1, Phường Phố Cò, Thành phố Sông Công, Tỉnh Thái Nguyên, Việt Nam","Công ty Cổ phần Đầu tư AHV Holding là một tổ chức đầu tư đa ngành, được thành lập với sứ mệnh kiến tạo hệ sinh thái doanh nghiệp vững mạnh, bền vững và có tính kết nối cao. Với nền tảng tài chính ..."
1,https://www.topcv.vn/brand/cennext/tuyen-dung/wordpress-developer-senior-level-20h-05h-remote-j1867723.html,None,WordPress Developer - Senior Level - (20h - 05h) Remote,Frontend Developer,12 - 18 triệu,Hà Nội và 3 nơi khác,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,NaN,3 năm,Cao Đẳng trở lên,Toàn thời gian,Nhân viên,3 người,"Build websites and landing pages on WordPress Customize and extend themes/plugins as required by each project. Publish content: create/edit Pages & Posts, set categories/tags, featured image, slug...",">3 years of hands-on WordPress publishing/site build experience. Good HTML/CSS/JS; PHP for theme/template tweaks. WordPress admin skills: Pages/Posts, Menus, Media, Users, Permalinks. Confident wi...","Competitive salary, negotiable based on capability. 85% salary during probation, with annual salary review. 13th-month salary, holiday bonuses, and Best Member awards. Training and development pro...",Chuyên môn Frontend Developer; IT - Phần mềm,05/04/2026,CENNEXT,https://cennext.com/,NaN,1000+,"Tầng 3, Nhà D, số 22 Thành Công, Ba Đình, Hà Nội","Được thành lập vào năm 2013, CENNEXT là một công ty chuyên về Dịch vụ xử lý dữ liệu từ xa trong lĩnh vực Thương mại điện tử (dịch thuật, quản lý website, facebook, chỉnh sửa photoshop...) cho các ..."
2,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,None,Data Processing Analyst,Data Analyst,25 - 30 triệu,Hồ Chí Minh (mới),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:",Thứ 2 - Thứ 6 (từ 09:00 đến 18:00),2 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,1 người,"Get to know the role We are looking for a new Analyst for one of our Data Processing Teams. We believe a successful candidate has problem solving and python programming skills, but if you believ

## 1. Clean cơ bản

In [85]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1477 entries, 0 to 1476
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   job_url                  1477 non-null   str   
 1   job_id                   0 non-null      object
 2   job_title_raw            1477 non-null   str   
 3   job_slug_raw             1477 non-null   str   
 4   salary_raw               1477 non-null   str   
 5   location_raw             1477 non-null   str   
 6   working_addresses_raw    1473 non-null   str   
 7   working_times_raw        1350 non-null   str   
 8   experience_raw           1477 non-null   str   
 9   education_level_raw      1474 non-null   str   
 10  employment_type_raw      1474 non-null   str   
 11  job_level_raw            1474 non-null   str   
 12  job_quantity_raw         1474 non-null   str   
 13  description_raw          1474 non-null   str   
 14  requirements_raw         1474 non-null   str   
 15

In [86]:
# Tạo các cột clean_* cho các cột text gốc
base_text_cols = [
    "job_title_raw",
    "job_slug_raw",
    "salary_raw",
    "location_raw",
    "working_addresses_raw",
    "working_times_raw",
    "experience_raw",
    "education_level_raw",
    "employment_type_raw",
    "job_level_raw",
    "job_quantity_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "tags_raw",
    "deadline_raw",
    "company_name_raw",
    "company_address_raw",
    "company_field_raw",
    "company_scale_raw",
    "company_description_raw",
]

for col in base_text_cols:
    if col in df_clean.columns:
        prefix = col.replace("_raw", "")
        df_clean[f"{prefix}_clean_light"] = df_clean[col].apply(clean_text_light)
        df_clean[f"{prefix}_clean_struct"] = df_clean[col].apply(clean_text_preserve_structure)
        df_clean[f"{prefix}_clean_strict"] = df_clean[col].apply(clean_text_strict)
        df_clean[f"{prefix}_clean_phobert"] = df_clean[col].apply(clean_text_for_phobert)

print("[INFO] Đã tạo xong các cột clean_*")
clean_cols = [c for c in df_clean.columns if "_clean_" in c]
print("Số cột clean:", len(clean_cols))
display(df_clean[[c for c in clean_cols[:12]]].head(5))

[INFO] Đã tạo xong các cột clean_*
Số cột clean: 84


,job_title_clean_light,job_title_clean_struct,job_title_clean_strict,job_title_clean_phobert,job_slug_clean_light,job_slug_clean_struct,job_slug_clean_strict,job_slug_clean_phobert,salary_clean_light,salary_clean_struct,salary_clean_strict,salary_clean_phobert
0,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,nhân viên tester mobile app android - đi làm ngay,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,Automation Tester,Automation Tester,automation tester,Automation Tester,Thoả thuận,Thoả thuận,thoả thuận,Thoả thuận
1,WordPress Developer - Senior Level - (20h - 05h) Remote,WordPress Developer - Senior Level - (20h - 05h) Remote,wordpress developer - senior level - 20h - 05h remote,WordPress Developer - Senior Level - (20h - 05h) Remote,Frontend Developer,Frontend Developer,frontend developer,Frontend Developer,12 - 18 triệu,12 - 18 triệu,12 - 18 triệu,12 - 18 triệu
2,Data Processing Analyst,Data Processing Analyst,data processing analyst,Data Processing Analyst,Data Analyst,Data Analyst,data analyst,Data Analyst,25 - 30 triệu,25 - 30 triệu,25 - 30 triệu,25 - 30 triệu
3,Fullstack Developer ( NodeJs ) - Yêu Cầu Kinh Nghiệm Từ 4 Năm Trở Lên (CoolMate),Fullstack Developer ( NodeJs ) - Yêu Cầu Kinh Nghiệm Từ 4 Năm Trở Lên (CoolMate),fullstack developer nodejs - yêu cầu kinh nghiệm từ 4 năm trở lên coolmate,Fullstack Developer ( NodeJs ) - Yêu Cầu Kinh Nghiệm Từ 4 Năm Trở Lên (CoolMate),Backend Developer | Fullstack Developer,Backend Developer | Fullstack Developer,backend developer fullstack developer,Backend Developer Fullstack Developer,20 - 25 triệu,20 - 25 triệu,20 - 25 triệu,20 - 25 triệu
4,Nhân Viên Backend Developer - Remote,Nhân Viên Backend Developer - Remote,nhân viên backend developer - remote,Nhân Viên Backend Developer - Remote,Backend Developer,Backend Developer,backend developer,Backend Developer,20 - 30 triệu,20 - 30 triệu,20 - 30 triệu,20 - 30 triệu


In [87]:
preview_cols = [
    # "job_title_raw", "job_title_clean_light", "job_title_clean_strict", "job_title_clean_phobert",
    # "job_slug_raw", "job_slug_clean_light", "job_slug_clean_strict", "job_slug_clean_phobert",
    # "requirements_raw", "requirements_clean_struct", "requirements_clean_strict", "requirements_clean_phobert",
    # "description_raw", "description_clean_struct","description_clean_light", "description_clean_phobert"
    "working_addresses_clean_light", "working_addresses_clean_struct", "working_addresses_clean_strict", "working_addresses_clean_phobert",
    "company_address_clean_light", "company_address_clean_struct", "company_address_clean_strict", "company_address_clean_phobert"
    
]
preview_cols = [c for c in preview_cols if c in df_clean.columns]

display(df_clean[preview_cols].head(3))

empty_ratio_df = pd.DataFrame({
    "column": preview_cols,
    "empty_ratio": [(df_clean[c].fillna("").str.strip() == "").mean() for c in preview_cols]
})
display(empty_ratio_df)

,working_addresses_clean_light,working_addresses_clean_struct,working_addresses_clean_strict,working_addresses_clean_phobert,company_address_clean_light,company_address_clean_struct,company_address_clean_strict,company_address_clean_phobert
0,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)","(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)",đã được cập nhật theo danh mục hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu - hà nội 116 nhân hòa thanh xuân phường thanh xuân quận thanh xuân cũ,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)","Số nhà 18, Tổ 1, Phường Phố Cò, Thành phố Sông Công, Tỉnh Thái Nguyên, Việt Nam","Số nhà 18, Tổ 1, Phường Phố Cò, Thành phố Sông Công, Tỉnh Thái Nguyên, Việt Nam",số nhà 18 tổ 1 phường phố cò thành phố sông công tỉnh thái nguyên việt nam,"Số nhà 18, Tổ 1, Phường Phố Cò, Thành phố Sông Công, Tỉnh Thái Nguyên, Việt Nam"
1,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,đã được cập nhật theo danh mục hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu - hà nội tất cả phường - hồ chí minh tất cả phường - đà nẵng tất cả phường - đồng nai tất cả phường ...,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,"Tầng 3, Nhà D, số 22 Thành Công, Ba Đình, Hà Nội","Tầng 3, Nhà D, số 22 Thành Công, Ba Đình, Hà Nội",tầng 3 nhà d số 22 thành công ba đình hà nội,"Tầng 3, Nhà D, số 22 Thành Công, Ba Đình, Hà Nội"
2,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:","(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:",đã được cập nhật theo danh mục hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu - hồ chí minh cmc creative space phường tân thuận quận 7 cũ hồ chí minh,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:","Tầng 4, 5, 6 và 7, QTSC Building 1, Lô 34, đường số 14, Công viên phần mềm Quang Trung, Phường Trung Mỹ Tây, TP Hồ Chí Minh","Tầng 4, 5, 6 và 7, QTSC Building 1, Lô 34, đường số 14, Công viên phần mềm Quang Trung, Phường Trung Mỹ Tây, TP Hồ Chí Minh",tầng 4 5 6 và 7 qtsc building 1 lô 34 đường số 14 công viên phần mềm quang trung phường trung mỹ tây tp hồ chí minh,"Tầng 4, 5, 6 và 7, QTSC Building 1, Lô 34, đường số 14, Công viên phần mềm Quang Trung, Phường Trung Mỹ Tây, TP Hồ Chí Minh"


,column,empty_ratio
0,working_addresses_clean_light,0.002708
1,working_addresses_clean_struct,0.002708
2,working_addresses_clean_strict,0.002708
3,working_addresses_clean_phobert,0.002708
4,company_address_clean_light,0.002031
5,company_address_clean_struct,0.002031
6,company_address_clean_strict,0.002031
7,company_address_clean_phobert,0.002031


In [88]:
# Tuỳ chọn hiển thị / debug
pd.set_option("display.max_rows", 1000)
# pd.set_option("display.max_columns", 200)
# pd.set_option("display.width", 200)
# pd.set_option("display.max_colwidth", 300)


### - Chuẩn hóa title

Notebook này đã được đồng bộ lại theo logic của file `.py` làm gốc.  
Phần title bên dưới giữ đúng luồng xử lý từ script: chuẩn hóa title, suy luận `job_family`, và tạo `job_title_for_phobert`.


In [89]:
# =========================
# TITLE PROCESSING - REVISED
# =========================

TITLE_SEGMENT_NOISE_PATTERNS = [
    # generic separators / garbage
    r"^\|.*$",
    r"^_+$",

    # salary / compensation / benefit
    r"^luong.*$",
    r"^thu nhap.*$",
    r"^salary.*$",
    r"^upto.*$",
    r"^up to.*$",
    r"^offer.*$",
    r"^phuc loi.*$",
    r"^benefit.*$",
    r"^gross.*$",
    r"^net.*$",
    r"^\d+\s*tr.*$",
    r"^\d+\s*trieu.*$",

    # location / workplace
    r"^ha noi.*$",
    r"^hn.*$",
    r"^ho chi minh.*$",
    r"^tp\.?\s*hcm.*$",
    r"^tphcm.*$",
    r"^hcm.*$",
    r"^sai gon.*$",
    r"^da nang.*$",
    r"^hai phong.*$",
    r"^can tho.*$",
    r"^bac ninh.*$",
    r"^thu duc.*$",
    r"^cau giay.*$",
    r"^dong da.*$",
    r"^quan \d+.*$",
    r"^tai ha noi.*$",
    r"^tai ho chi minh.*$",
    r"^tai tp\.?\s*hcm.*$",
    r"^lam viec tai.*$",

    # work mode / urgency / shift
    r"^di lam ngay.*$",
    r"^ob som.*$",
    r"^remote.*$",
    r"^hybrid.*$",
    r"^onsite.*$",
    r"^urgent.*$",
    r"^hot.*$",
    r"^\d{1,2}h\s*-\s*\d{1,2}h.*$",

    # org / department noise
    r"^khoi .*",
    r"^phong .*",
    r"^ban .*",
    r"^team .*",
    r"^du an .*",
    r"^mang .*",
    r"^domain .*",

    # internal code
    r"^ta\d+.*$",
    r"^ho\d{2}\.\d+.*$",
    r"^holt\.\d+.*$",
    r"^it\d+.*$",
    r"^\d+\..*$",

    # experience / requirement
    r"^tu\s*\d+\s*nam.*$",
    r"^\d+\+?\s*nam.*$",
    r"^\d+\s*nam kinh nghiem.*$",
    r"^kinh nghiem.*$",
    r"^yeu cau.*$",
    r"^khong yeu cau kinh nghiem.*$",
    r"^at least.*$",
    r"^english required.*$",
    r"^good at english.*$",
    r"^tieng anh.*$",
    r"^jlpt.*$",
    r"^n\d\+?.*$",

    # company / branding / contextual tails
    r"^funtap.*$",
    r"^coolmate.*$",
    r"^misa.*$",
    r"^banking project.*$",
    r"^healthcare.*$",
    r"^bank.*$",
    r"^fintech.*$",
    r"^edtech.*$",
    r"^ecommerce.*$",
    r"^mobile app.*$",
    r"^web app.*$",

    # standalone seniority / working form in brackets or split tails
    r"^intern$",
    r"^junior$",
    r"^middle$",
    r"^senior$",
    r"^lead$",
    r"^leader$",
    r"^team lead$",
    r"^fresher$",
    r"^fulltime$",
    r"^full time$",
    r"^m/f/d$",
    r"^all levels$",
]

BRACKET_NOISE_KEYWORDS = {
    # location
    "ha noi", "hn", "ho chi minh", "hcm", "tp hcm", "tphcm", "sai gon",
    "da nang", "hai phong", "can tho", "bac ninh", "thu duc", "cau giay", "dong da",

    # work mode / urgency
    "remote", "hybrid", "onsite", "urgent", "hot", "di lam ngay", "ob som",

    # internal codes
    "ta105", "ta150", "ta171", "ta172", "ta174", "ta188",
    "ho26.37", "ho26.69", "ho26.74", "ho26.76", "ho26.77", "ho26.78", "ho26.79", "ho26.84", "ho26.85", "ho26.95", "ho26.96",
    "holt.03", "holt.04", "holt.05", "holt.07", "holt.08", "holt.09", "holt.11", "holt.12", "holt.13", "holt.14", "holt.15", "holt.16", "holt.17", "holt.18", "holt.19",

    # standalone seniority / working form
    "intern", "junior", "middle", "senior", "lead", "leader", "team lead", "fresher", "fulltime", "full time", "m/f/d", "all levels",

    # generic contextual noise in brackets
    "banking", "fintech", "game industry", "domain erp", "domain edtech",
    "salary upto 25m", "salary upto 35m", "salary upto 50m",
}

# IMPORTANT:
# Keys in TITLE_SYNONYM_MAP are stored in normalize_for_match() form.
# Canonical values are intentionally compact/stable for grouping and downstream analysis.
TITLE_SYNONYM_MAP = {
    # =====================
    # DATA ANALYST
    # =====================
    "data analyst": "data analyst",
    "senior data analyst": "data analyst",
    "middle data analyst": "data analyst",
    "junior data analyst": "data analyst",
    "chuyen vien phan tich du lieu": "data analyst",
    "nhan vien phan tich du lieu": "data analyst",
    "phan tich du lieu": "data analyst",
    "phan tich du lieu kinh doanh": "data analyst",
    "chuyen vien phan tich va thiet ke mo hinh du lieu": "data analyst",
    "chuyen vien phan tich du lieu nguoi dung": "data analyst",
    "business data analyst": "data analyst",
    "business customer data analyst": "data analyst",
    "customer data analyst": "data analyst",
    "data analysis executive": "data analyst",
    "assistant manager data analyst": "data analyst",
    "strategic data lead": "data analyst",
    "data analyst teamleader": "data analyst",
    "customer data analyst team lead": "data analyst",
    "hr data analysis": "data analyst",
    "ecommerce business data analyst": "data analyst",
    "cvcc phan tich du lieu": "data analyst",
    "chuyen vien du lieu tai chinh": "data analyst",
    "chuyen vien phan tich va quan ly du lieu tai chinh": "data analyst",
    "credit risk analytics and modelling expert": "data analyst",
    "expert fraud risk data analytics and portfolio management": "data analyst",
    "fp&a analyst": "data analyst",
    "finance planning & analysis associate": "data analyst",
    "junior fp&a analyst": "data analyst",
    "bi analyst": "data analyst",
    "power bi leader": "data analyst",
    "presale data and analytics": "data analyst",

    # =====================
    # DATA ENGINEER
    # =====================
    "data engineer": "data engineer",
    "fresher data engineer": "data engineer",
    "thuc tap sinh data engineer": "data engineer",
    "data engineer intern": "data engineer",
    "nhan vien data engineer": "data engineer",
    "nhan vien data engineering": "data engineer",
    "chuyen vien du lieu data engineer": "data engineer",
    "ky su du lieu": "data engineer",
    "ky su du lieu lon": "data engineer",
    "big data engineer": "data engineer",
    "big data admin": "data engineer",
    "data integration engineer": "data engineer",
    "chuyen vien data integration engineer": "data engineer",
    "junior data integration engineer": "data engineer",
    "data warehouse engineer": "data engineer",
    "fresher data warehouse": "data engineer",
    "fresher r&d engineer data warehouse": "data engineer",
    "data platform engineer": "data engineer",
    "data platform operation": "data engineer",
    "data center storage engineer": "data engineer",
    "data engineer aws": "data engineer",
    "aws data engineer": "data engineer",
    "data engineer java python scala hadoop kafka": "data engineer",
    "data management": "data engineer",
    "chuyen vien du lieu": "data engineer",
    "chuyen gia du lieu": "data engineer",
    "gsd engineer garment standard data engineer ky su he thong du lieu chuan": "data engineer",

    # =====================
    # DATA SCIENTIST
    # =====================
    "data scientist": "data scientist",
    "senior data scientist": "data scientist",
    "data scientist expert": "data scientist",
    "nha khoa hoc du lieu": "data scientist",
    "cvcc khoa hoc du lieu": "data scientist",
    "chuyen vien phat trien khoa hoc du lieu": "data scientist",
    "chuyen vien mo hinh va phan tich nang cao": "data scientist",
    "chuyen vien cao cap mo hinh hoa va phan tich nang cao": "data scientist",
    "chuyen vien chuyen vien cao cap khoa hoc du lieu": "data scientist",
    "chuyen vien phat trien khoa hoc du lieu data scientist": "data scientist",
    "data science intern": "data scientist",
    "data scientist python llm ai machine learning": "data scientist",

    # =====================
    # AI ENGINEER
    # =====================
    "ai engineer": "ai engineer",
    "ai engineer intern": "ai engineer",
    "ctv ai engineer": "ai engineer",
    "ai developer": "ai engineer",
    "ai software engineer": "ai engineer",
    "ai software engineer ky su phan mem ai": "ai engineer",
    "ai engineering": "ai engineer",
    "ai generative engineer": "ai engineer",
    "ai machine learning engineer": "ai engineer",
    "machine learning engineer": "ai engineer",
    "ml engineer": "ai engineer",
    "agentic engineer": "ai engineer",
    "ai agent engineer": "ai engineer",
    "edge ai engineer": "ai engineer",
    "edge ai engineer leader": "ai engineer",
    "ai platform engineer": "ai engineer",
    "ai platform architect": "ai engineer",
    "ai system engineer": "ai engineer",
    "nhan vien ai system engineer": "ai engineer",
    "ky su ai": "ai engineer",
    "ky su tri tue nhan tao": "ai engineer",
    "ky su tri tue nhan tao ai": "ai engineer",
    "ky su ai ai engineer": "ai engineer",
    "ky su phat trien ai ai engineer": "ai engineer",
    "chuyen vien tri tue nhan tao": "ai engineer",
    "chuyen vien tri tue nhan tao ai": "ai engineer",
    "chuyen vien ai": "ai engineer",
    "chuyen vien ai ai engineer": "ai engineer",
    "lap trinh vien ai": "ai engineer",
    "fresher ai": "ai engineer",
    "fresher ai nlp": "ai engineer",
    "fresher ai computer vision engineer": "ai engineer",
    "aiops specialist": "ai engineer",
    "prompt engineer": "ai engineer",
    "software engineer prompt engineering": "ai engineer",
    "ai engineer computer vision npl llm": "ai engineer",
    "ai engineer nlp": "ai engineer",
    "ai engineer llm": "ai engineer",
    "ai engineer generative ai llm": "ai engineer",
    "ai engineer rag langchain python n8n": "ai engineer",
    "ai engineer python microservices docker": "ai engineer",
    "ai engineer healthcare ai product": "ai engineer",
    "ai engineer ky su ai": "ai engineer",
    "ai native software engineering lead": "ai engineer",
    "mid level ai powered cross platform engineer": "ai engineer",
    "kỹ sư cloud ai": "ai engineer",
    "chuyen vien lap trinh iot ai": "ai engineer",
    "chuyen vien phat trien ai app tu dong hoa": "ai engineer",
    "chuyen vien phat trien du an ai": "ai engineer",
    "nhan vien phat trien giai phap ai marketing": "ai engineer",
    "product & ai automation intern": "ai engineer",
    "ai and automation intern": "ai engineer",
    "mb trainee ai engineer": "ai engineer",
    "on job training ai": "ai engineer",
    "intern ai solution engineer": "ai engineer",

    # =====================
    # AI RESEARCHER
    # =====================
    "ai researcher": "ai researcher",
    "ai research engineer": "ai researcher",
    "senior ai research engineer": "ai researcher",
    "nlp research engineer": "ai researcher",
    "ai research intern": "ai researcher",
    "ai quantitative researcher intern": "ai researcher",
    "nhan vien nghien cuu al ai researcher": "ai researcher",
    "thuc tap sinh nghien cuu ung dung ai": "ai researcher",

    # =====================
    # DATA LABELING
    # =====================
    "data labeling specialist": "data labeling specialist",
    "data labeling coordinator": "data labeling specialist",
    "dieu phoi du an label data tieng anh": "data labeling specialist",
    "thuc tap sinh xu ly du lieu": "data labeling specialist",
    "data processing analyst": "data labeling specialist",
    "data processing specialist": "data labeling specialist",
    "chuyen vien xu ly du lieu": "data labeling specialist",
    "nhan vien xu ly du lieu": "data labeling specialist",
    "nhan vien xu ly du lieu team scan kols shopee": "data labeling specialist",
    "data entry specialist": "data labeling specialist",
    "nhan vien nhap lieu xu ly du lieu": "data labeling specialist",
    "nhan vien nhap va xu ly du lieu tieng nhat": "data labeling specialist",
    "nhan vien gan nhan du lieu tieng nhat": "data labeling specialist",
    "nhan vien ngon ngu du lieu tieng anh": "data labeling specialist",
    "nhan vien ngon ngu du lieu tieng trung": "data labeling specialist",
    "nhan vien ngon ngu du lieu tieng phap": "data labeling specialist",
    "nhan vien ngon ngu du lieu tieng tay ban nha bo dao nha thai": "data labeling specialist",
    "language data specialist": "data labeling specialist",
    "cong tac vien xu ly du lieu hinh anh lam fulltime offline": "data labeling specialist",

    # =====================
    # SOFTWARE ENGINEER
    # =====================
    "software engineer": "software engineer",
    "java software engineer": "software engineer",
    "bridge software engineer": "software engineer",
    "lead software engineer desktop": "software engineer",
    "middle software engineer java net frontend": "software engineer",
    "lap trinh vien phat trien phan mem software engineer": "software engineer",
    "laup trinh vien software engineer it": "software engineer",
    "ky su giai phap phan mem software developer": "software engineer",
    "ky su phat trien phan mem": "software engineer",
    "chuyen vien phat trien phan mem": "software engineer",
    "chuyen vien phat trien ung dung": "software engineer",
    "nhan vien software developer backend dev": "software engineer",
    "nhan vien software developer froned dev": "software engineer",
    "nhan vien trien khai phan mem mes software engineer": "software engineer",

    # =====================
    # BACKEND DEVELOPER
    # =====================
    "backend developer": "backend developer",
    "backend engineer": "backend developer",
    "backend lead engineer": "backend developer",
    "backend technical lead": "backend developer",
    "backend middle developer": "backend developer",
    "middle backend developer": "backend developer",
    "middle back end engineer": "backend developer",
    "middle senior backend engineer": "backend developer",
    "nhan vien backend": "backend developer",
    "nhan vien backend engineer developer": "backend developer",
    "nhan vien lap trinh backend": "backend developer",
    "lap trinh vien backend": "backend developer",
    "lap trinh vien backend developer java": "backend developer",
    "lap trinh vien backend python": "backend developer",
    "lap trinh vien backend net java nodejs php": "backend developer",
    "lap trinh vien back end developer": "backend developer",
    "java developer": "backend developer",
    "java engineer": "backend developer",
    "java backend": "backend developer",
    "java backend developer": "backend developer",
    "java back end developer": "backend developer",
    "java backend senior tech lead": "backend developer",
    "middle java backend": "backend developer",
    "middle java backend developer": "backend developer",
    "java developer backend engineer": "backend developer",
    "python developer": "backend developer",
    "junior python developer": "backend developer",
    "nodejs developer": "backend developer",
    "middle node js developer": "backend developer",
    "php developer": "backend developer",
    "php developer laravel": "backend developer",
    "php backend developer": "backend developer",
    "php teamleader": "backend developer",
    "golang developer": "backend developer",
    "golang backend": "backend developer",
    "lap trinh vien golang": "backend developer",
    ".net developer": "backend developer",
    ". net developer": "backend developer",
    "dotnet developer": "backend developer",
    "senior .net developer": "backend developer",
    "junior .net c#": "backend developer",
    "lap trinh vien net": "backend developer",
    "lap trinh vien .net": "backend developer",
    "lap trinh vien .net back end": "backend developer",
    "lap trinh vien .net leader": "backend developer",
    "nhan vien lap trinh .net": "backend developer",
    "nhan vien lap trinh vien backend php": "backend developer",
    "back end developer": "backend developer",
    "backend developer spring boot golang": "backend developer",
    "backend developer golang": "backend developer",
    "backend developer java": "backend developer",
    "backend developer javascript python": "backend developer",
    "backend developer nodejs": "backend developer",
    "backend developer python dev": "backend developer",
    "backend developer asp net core": "backend developer",
    "backend developer .net": "backend developer",
    "backend engineer .net java": "backend developer",

    # =====================
    # FRONTEND DEVELOPER
    # =====================
    "frontend developer": "frontend developer",
    "frontend engineer": "frontend developer",
    "front end developer": "frontend developer",
    "front end engineer": "frontend developer",
    "front end lead": "frontend developer",
    "front end team lead": "frontend developer",
    "junior frontend developer": "frontend developer",
    "middle frontend developer": "frontend developer",
    "middle senior frontend engineer": "frontend developer",
    "react native developer": "frontend developer",
    "reactjs developer": "frontend developer",
    "frontend reactjs developer": "frontend developer",
    "frontend reactjs nextjs": "frontend developer",
    "react native": "frontend developer",
    "angular developer": "frontend developer",
    "chuyen vien lap trinh angular": "frontend developer",
    "lap trinh vien angular": "frontend developer",
    "vuejs developer": "frontend developer",
    "frontend developer nextjs": "frontend developer",
    "lap trinh vien frontend": "frontend developer",
    "lap trinh vien giao dien frontend developer": "frontend developer",
    "lap trinh vien web front end reactjs nextjs": "frontend developer",
    "nhan vien lap trinh frontend": "frontend developer",
    "nhan vien phat trien phan mem frontend developer": "frontend developer",
    "nhan vien phan mem frontend": "frontend developer",

    # =====================
    # FULLSTACK DEVELOPER
    # =====================
    "fullstack developer": "fullstack developer",
    "full stack developer": "fullstack developer",
    "full stack engineer": "fullstack developer",
    "fullstack engineer": "fullstack developer",
    "fullstack software engineer": "fullstack developer",
    "dev nodejs fullstack": "fullstack developer",
    "fullstack java developer": "fullstack developer",
    "java fullstack developer": "fullstack developer",
    "fullstack c# .net": "fullstack developer",
    "talent developer fullstack": "fullstack developer",
    "middle full stack developer": "fullstack developer",
    "middle fullstack developer": "fullstack developer",
    "middle senior full stack developer python": "fullstack developer",
    "nhan vien lap trinh full stack developer": "fullstack developer",
    "nhan vien lap trinh fullstack developer": "fullstack developer",
    "nhan vien lap trinh fullstack": "fullstack developer",
    "lap trinh vien fullstack": "fullstack developer",
    "lap trinh vien full stack": "fullstack developer",
    "lap trinh vien full stack desktop web mobile": "fullstack developer",
    "lap trinh fullstack": "fullstack developer",
    "chuyen vien phat trien phan mem fullstack": "fullstack developer",
    "chuyen vien phat trien phan mem fullstack": "fullstack developer",
    "chuyen vien phat trien phan mem fullstack": "fullstack developer",

    # =====================
    # CLOUD ENGINEER
    # =====================
    "cloud engineer": "cloud engineer",
    "junior cloud engineer": "cloud engineer",
    "aws cloud lead": "cloud engineer",
    "azure api engineer": "cloud engineer",
    "cloud support engineer": "cloud engineer",
    "post sales system & cloud engineer": "cloud engineer",
    "ky su giai phap cloud": "cloud engineer",
    "ky su giai phap dien toan dam may senior cloud engineer": "cloud engineer",
    "ky su he thong cloud engineer": "cloud engineer",
    "it infrastructure system network & cloud": "cloud engineer",
    "cloud engineer aws": "cloud engineer",
    "cloud engineer aws oci azure": "cloud engineer",
    "cloud engineer storage": "cloud engineer",
    "data center storage engineer": "cloud engineer",

    # =====================
    # DATABASE ADMINISTRATOR
    # =====================
    "database administrator": "database administrator",
    "database administrator dba": "database administrator",
    "senior database administrator": "database administrator",
    "dba": "database administrator",
    "quan tri co so du lieu": "database administrator",
    "nhan vien quan tri co so du lieu dba": "database administrator",
    "nhan su quan tri co so du lieu dba": "database administrator",
    "database engineer dba": "database administrator",

    # =====================
    # AUTOMATION TESTER
    # =====================
    "automation tester": "automation tester",
    "automation test engineer": "automation tester",
    "automation test": "automation tester",
    "automation testing": "automation tester",
    "performance tester": "automation tester",
    "manual tester": "automation tester",
    "manual tester junior": "automation tester",
    "manual tester intern": "automation tester",
    "senior tester": "automation tester",
    "middle tester": "automation tester",
    "junior tester": "automation tester",
    "tester": "automation tester",
    "tester product": "automation tester",
    "app mobile tester": "automation tester",
    "intern tester": "automation tester",
    "intern tester tieng nhat": "automation tester",
    "qc tester": "automation tester",
    "uat tester": "automation tester",
    "qa tester": "automation tester",
    "software tester": "automation tester",
    "nhan vien tester": "automation tester",
    "nhan vien tester mobile app": "automation tester",
    "nhan vien kiem thu phan mem": "automation tester",
    "nhan vien kiem thu phan mem software qa": "automation tester",
    "nhan vien kiem thu tester": "automation tester",
    "chuyen vien kiem thu phan mem": "automation tester",
    "chuyen vien kiem thu phan mem auto": "automation tester",
    "chuyen vien kiem thu phan mem manual tester agile": "automation tester",
    "chuyen vien kiem thu phan mem tester": "automation tester",
    "chuyen vien kiem tra chat luong phan mem tester": "automation tester",
    "kiem thu phan mem": "automation tester",
    "kiem thu phan mem tester": "automation tester",
    "ai augmented quality engineer tester": "automation tester",
    "ai quality assurance automation focus": "automation tester",
    "junior qc japanese n4 kiem thu phan mem": "automation tester",

    # =====================
    # DEVOPS ENGINEER
    # =====================
    "devops engineer": "devops engineer",
    "devops": "devops engineer",
    "devops cloud engineer": "devops engineer",
    "senior devops engineer": "devops engineer",
    "devops engineer senior": "devops engineer",
    "devops engineer site reliability engineer": "devops engineer",
    "devops engineer sysops engineer": "devops engineer",
    "devops engineer kubernetes cloud": "devops engineer",
    "devops engineer azure": "devops engineer",
    "devops engineer middle": "devops engineer",
    "devops engineer junior": "devops engineer",
    "fresher devops": "devops engineer",
    "thuc tap sinh devops": "devops engineer",
    "ky su devops": "devops engineer",
    "ky su van hanh va phat trien he thong devops": "devops engineer",
    "ky su phat trien van hanh he thong devops": "devops engineer",
    "nhan vien van hanh ung dung devops": "devops engineer",
    "cloud devops engineer": "devops engineer",
    "cloud devops engineer cds": "devops engineer",
    "devsecops engineer": "devops engineer",
    "appsec devops devsecops engineer": "devops engineer",
    "site reliability engineer sre trong devops team": "devops engineer",
    "chuyen vien sre": "devops engineer",

    # =====================
    # IOT ENGINEER
    # =====================
    "iot engineer": "iot engineer",
    "ky su iot": "iot engineer",
    "ki su iot": "iot engineer",
    "ky su iot iot engineer": "iot engineer",
    "chuyen vien iot iot platform digital twin": "iot engineer",
    "chuyen vien ky thuat iot": "iot engineer",
    "nhan vien ky thuat iot": "iot engineer",
    "nhan vien thiet bi iot": "iot engineer",
    "chuyen vien ky thuat thiet bi dien thoai iot": "iot engineer",
    "ky su tich hop he thong iot va lap trinh nhung": "iot engineer",
    "ky su nhung embedded iot": "iot engineer",
    "ky su nhung iot": "iot engineer",

    # =====================
    # EMBEDDED ENGINEER
    # =====================
    "embedded engineer": "embedded engineer",
    "embedded developer": "embedded engineer",
    "embedded software engineer": "embedded engineer",
    "embedded software development": "embedded engineer",
    "embedded linux engineer": "embedded engineer",
    "embedded systems engineer": "embedded engineer",
    "embedded android developer": "embedded engineer",
    "fresher embedded developer": "embedded engineer",
    "firmware engineer": "embedded engineer",
    "embedded software engineer lead": "embedded engineer",
    "embedded software verification and validation": "embedded engineer",
    "ky su he thong nhung": "embedded engineer",
    "ky su lap trinh nhung": "embedded engineer",
    "ky su lap trinh embedded linux": "embedded engineer",
    "ky su lap trinh firmware": "embedded engineer",
    "ky su phan mem nhung": "embedded engineer",
    "lap trinh nhung": "embedded engineer",
    "lap trinh phan mem nhung": "embedded engineer",
    "lap trinh vien nhung embedded": "embedded engineer",
    "embedded engineer lap trinh nhung": "embedded engineer",
    "ky su chuyen vien phan mem nhung": "embedded engineer",

    # =====================
    # PRODUCT ANALYST
    # =====================
    "product analyst": "product analyst",
    "product analyst research": "product analyst",
    "data product analyst": "product analyst",
    "product executive": "product analyst",
    "product development executive": "product analyst",
    "product management executive": "product analyst",
    "product management specialist": "product analyst",
    "product operator": "product analyst",
    "product operations executive": "product analyst",
    "assistant product manager": "product analyst",
    "business product lead": "product analyst",

    # =====================
    # BUSINESS ANALYST
    # =====================
    "business analyst": "business analyst",
    "it business analyst": "business analyst",
    "junior business analyst": "business analyst",
    "middle business analyst": "business analyst",
    "senior business analyst": "business analyst",
    "business analyst leader": "business analyst",
    "business analyst lead": "business analyst",
    "business analyst intern": "business analyst",
    "business analyst internship": "business analyst",
    "business analyst fresher": "business analyst",
    "business analyst data": "business analyst",
    "data business analyst": "business analyst",
    "erp business analyst": "business analyst",
    "ba business analyst": "business analyst",
    "chuyen vien business analyst": "business analyst",
    "chuyen vien phan tich nghiep vu": "business analyst",
    "nhan vien phan tich nghiep vu": "business analyst",
    "phan tich nghiep vu": "business analyst",
    "phan tich nghiep vu business analyst": "business analyst",
    "chuyen vien phan tich nghiep vu business analyst": "business analyst",
    "chuyen vien phan tich nghiep vu ba": "business analyst",
    "nhan vien phan tich nghiep vu ba": "business analyst",
    "business analyst chuyen vien phan tich nghiep vu": "business analyst",
    "business analyst ba": "business analyst",
    "nhan vien business analyst": "business analyst",
    "nhan vien ba": "business analyst",
    "nhan vien ba business analyst": "business analyst",
    "thuc tap sinh phan tich nghiep vu": "business analyst",
    "thuc tap sinh phan tich nghiep vu business analyst intern": "business analyst",
    "junior ba phan tich yeu cau quy trinh nghiep vu": "business analyst",
    "leader ba it": "business analyst",
    "chuyen vien phan tich he thong nghiep vu": "business analyst",
    "business analyst crm": "business analyst",
    "business analyst erp lead": "business analyst",
    "chuyen vien ba erp tu van trien khai erp phan mem ke toan": "business analyst",
    "chuyen vien phan tich nghiep vu odoo": "business analyst",
    "business analyst banking fintech": "business analyst",
    "business analyst domain bank fintech": "business analyst",

    # =====================
    # PRODUCT OWNER / PRODUCT MANAGER
    # =====================
    "product owner": "product manager",
    "product manager": "product manager",
    "nhan vien product owner": "product manager",
    "product owner po": "product manager",
    "product owner product manager": "product manager",
    "product manager product owner": "product manager",
    "product executive product owner": "product manager",
    "giam doc san pham product manager": "product manager",
    "chuyen gia phat trien san pham po": "product manager",
    "chuyen gia san pham": "product manager",
    "chuyen gia phat trien san pham so khcn product owner": "product manager",
    "fresher product owner": "product manager",
    "intern product owner": "product manager",
    "product owner intern": "product manager",
    "game product owner": "product manager",
    "product owner mobile app": "product manager",
    "product owner rpa product": "product manager",
    "product owner edtech ai game": "product manager",
    "product owner fintech": "product manager",
    "product owner expert": "product manager",
    "product manager ecommerce": "product manager",
    "product manager erp logistics wms": "product manager",
    "product manager mobile app": "product manager",
    "product manager technical solution": "product manager",
}

TITLE_CANONICAL_KEYWORDS = {
    "data analyst": [
        "data analyst", "business data analyst", "bi analyst", "fp&a analyst", "hr data analysis",
        "credit risk analytics", "fraud risk data analytics", "phan tich du lieu",
    ],
    "data engineer": [
        "data engineer", "data integration engineer", "data warehouse engineer", "data platform engineer",
        "big data", "data crawler engineer", "ky su du lieu",
    ],
    "data scientist": [
        "data scientist", "nha khoa hoc du lieu", "khoa hoc du lieu",
    ],
    "ai engineer": [
        "ai engineer", "machine learning engineer", "ml engineer", "ai developer", "agentic engineer",
        "computer vision", "nlp engineer", "ai platform", "prompt engineer",
    ],
    "ai researcher": [
        "ai researcher", "ai research", "research engineer", "nlp research",
    ],
    "data labeling specialist": [
        "data labeling", "label data", "gan nhan du lieu", "xu ly du lieu", "ngon ngu du lieu", "nhap lieu",
    ],
    "software engineer": [
        "software engineer", "software developer", "phat trien phan mem", "phat trien ung dung",
    ],
    "backend developer": [
        "backend developer", "backend engineer", "java developer", "java backend", "python developer",
        "nodejs developer", "php developer", "golang developer", ".net developer", "dotnet developer",
    ],
    "frontend developer": [
        "frontend developer", "frontend engineer", "angular developer", "reactjs developer",
        "react native developer", "vuejs developer", "front end developer",
    ],
    "fullstack developer": [
        "fullstack developer", "full stack developer", "fullstack engineer", "full stack engineer",
    ],
    "cloud engineer": [
        "cloud engineer", "aws cloud", "azure api engineer", "cloud support",
    ],
    "database administrator": [
        "database administrator", "dba", "quan tri co so du lieu",
    ],
    "automation tester": [
        "automation tester", "automation test", "performance tester", "manual tester", "tester",
        "software tester", "qa tester", "uat tester", "kiem thu phan mem",
    ],
    "devops engineer": [
        "devops engineer", "devsecops engineer", "site reliability engineer", "sre", "sysops",
    ],
    "iot engineer": [
        "iot engineer", "ky su iot", "ki su iot", "thiet bi iot",
    ],
    "embedded engineer": [
        "embedded engineer", "embedded developer", "embedded software", "firmware engineer",
        "lap trinh nhung", "he thong nhung",
    ],
    "product analyst": [
        "product analyst", "product executive", "product operations", "assistant product manager",
    ],
    "business analyst": [
        "business analyst", "phan tich nghiep vu", "ba", "it business analyst", "erp business analyst",
    ],
    "product manager": [
        "product owner", "product manager", "po",
    ],
}

JOB_FAMILY_RULES = {
    "data_analytics": [
        "data analyst", "product analyst",
    ],
    "data_engineering": [
        "data engineer", "database administrator",
    ],
    "data_science_ml": [
        "data scientist", "ai engineer", "ai researcher",
    ],
    "software_engineering": [
        "software engineer", "backend developer", "frontend developer", "fullstack developer",
        "embedded engineer", "iot engineer",
    ],
    "cloud_devops_sre": [
        "cloud engineer", "devops engineer",
    ],
    "qa_testing": [
        "automation tester",
    ],
    "product_project_ba": [
        "business analyst", "product manager", "product analyst",
    ],
    "data_governance_quality": [
        "data labeling specialist",
    ],
    "database_platform": [
        "database administrator",
    ],
}

JOB_FAMILY_DESCRIPTION_HINTS = {
    "data_analytics": [
        "dashboard", "report", "business intelligence", "power bi", "tableau", "sql", "kpi", "insight",
        "phan tich du lieu",
    ],
    "data_engineering": [
        "etl", "pipeline", "data pipeline", "data warehouse", "dwh", "airflow", "spark",
        "kafka", "hadoop", "lakehouse", "fabric", "batch", "streaming", "database administrator",
    ],
    "data_science_ml": [
        "machine learning", "deep learning", "nlp", "llm", "rag", "computer vision", "model training",
        "tensorflow", "pytorch", "generative ai", "genai", "feature engineering",
    ],
    "software_engineering": [
        "software engineer", "backend developer", "frontend developer", "fullstack developer",
        "backend", "frontend", "fullstack", "api", "microservice", "rest", "graphql",
        "react", "nodejs", "java", ".net", "web development", "wordpress",
        "embedded", "firmware", "iot",
    ],
    "cloud_devops_sre": [
        "cloud engineer", "devops engineer",
        "cloud", "aws", "azure", "gcp", "kubernetes", "docker", "terraform", "ci/cd",
        "monitoring", "observability", "infrastructure as code",
    ],
    "qa_testing": [
        "automation tester", "manual tester", "software tester",
        "automation test", "test automation", "qa", "quality assurance", "selenium", "cypress",
        "playwright", "regression test", "performance test",
    ],
    "product_project_ba": [
        "product analyst", "business analyst", "phan tich nghiep vu",
        "product owner", "product manager",
        "user story", "brd", "frd", "stakeholder", "scrum", "agile",
        "project management", "product roadmap", "backlog", "prioritization",
    ],
    "data_governance_quality": [
        "data governance", "data quality", "master data", "data steward", "label", "labeling",
        "gan nhan", "data validation", "metadata", "lineage", "xu ly du lieu", "nhap lieu",
    ],
    "database_platform": [
        "database administrator", "dba",
        "database", "sql server", "postgresql", "mysql", "query tuning", "index", "backup", "replication",
    ],
}

DISPLAY_REPLACEMENTS = [
    (r"\bBA\b", "Business Analyst"),
    (r"\bPO\b", "Product Owner"),
    (r"\bPM\b", "Project Manager"),
    (r"\bQA\b", "QA"),
    (r"\bQC\b", "QC"),
    (r"\bAI\b", "AI"),
    (r"\bNLP\b", "NLP"),
    (r"\bLLM\b", "LLM"),
    (r"\bRAG\b", "RAG"),
    (r"\bIoT\b", "IoT"),
    (r"\bDBA\b", "DBA"),
    (r"\bDevOps\b", "DevOps"),
    (r"\bNodejs\b", "NodeJS"),
    (r"\bReactjs\b", "ReactJS"),
    (r"\bVuejs\b", "VueJS"),
    (r"\bDotnet\b", ".NET"),
]

def strip_bracket_noise(text):
    text = safe_str(text)
    if not text:
        return ""

    matches_round = re.findall(r"\((.*?)\)", text)
    for m in matches_round:
        normalized = normalize_for_match(m)
        if normalized in BRACKET_NOISE_KEYWORDS:
            text = text.replace(f"({m})", " ")

    matches_square = re.findall(r"\[(.*?)\]", text)
    for m in matches_square:
        normalized = normalize_for_match(m)
        if normalized in BRACKET_NOISE_KEYWORDS:
            text = text.replace(f"[{m}]", " ")

    return re.sub(r"\s+", " ", text).strip()

def remove_internal_title_codes(text):
    text = safe_str(text)
    if not text:
        return ""
    text = re.sub(r"\b(?:TA\d+|HO\d{2}\.\d+|HOLT\.\d+|IT\d+)\b", " ", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip(" -|_")
    return text

def split_title_segments(text):
    text = safe_str(text)
    if not text:
        return []
    parts = [p.strip() for p in re.split(r"\s*-\s*", text) if p.strip()]
    return parts

def normalize_segment_for_dedup(segment):
    s = normalize_for_match(segment)
    s = re.sub(r"\b(junior|middle|senior|lead|leader|team lead|fresher|intern)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_noise_segment(segment):
    s = normalize_for_match(segment)
    if not s:
        return True
    for pat in TITLE_SEGMENT_NOISE_PATTERNS:
        if re.search(pat, s, flags=re.I):
            return True
    return False

def should_keep_bracket_content(content):
    s = normalize_for_match(content)
    if not s:
        return False

    # Drop purely noisy brackets
    if s in BRACKET_NOISE_KEYWORDS:
        return False

    # Keep meaningful technical/specialization cues
    keep_keywords = [
        "business analyst", "ba", "product owner", "product manager",
        "python", "java", "golang", "go", "node", "nodejs", "php", "ruby",
        "react", "reactjs", "react native", "angular", "vue", "vuejs", "nextjs",
        ".net", "net", "c#", "c++", "kotlin", "swift",
        "aws", "azure", "gcp", "cloud", "kubernetes", "docker",
        "nlp", "llm", "rag", "computer vision", "cv", "agentic", "generative ai",
        "automation", "manual tester", "tester", "dba", "mysql", "sql", "spring boot",
        "embedded", "firmware", "iot",
    ]
    return any(k in s for k in keep_keywords)

def clean_bracket_content(text):
    text = safe_str(text)
    if not text:
        return ""

    def _replace_round(match):
        content = match.group(1).strip()
        return f"({content})" if should_keep_bracket_content(content) else " "

    def _replace_square(match):
        content = match.group(1).strip()
        return f"[{content}]" if should_keep_bracket_content(content) else " "

    text = re.sub(r"\((.*?)\)", _replace_round, text)
    text = re.sub(r"\[(.*?)\]", _replace_square, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_title_global_noise(text):
    text = safe_str(text)
    if not text:
        return ""

    normalized = normalize_for_match(text)

    # remove common noisy long tails directly on raw text by patterns aligned to normalized content
    replacements = [
        (r"(?i)\b(?:lương|thu nhập|salary|upto|up to|offer)\b.*$", ""),
        (r"(?i)\b(?:đi làm ngay|ob sớm|urgent|hot)\b.*$", ""),
        (r"(?i)\b(?:từ|from)\s*\d+\s*năm.*$", ""),
        (r"(?i)\b\d+\s*năm\s*kinh\s*nghiệm.*$", ""),
        (r"(?i)\b(?:english required|good at english|tiếng anh tốt|n\d\+|jlpt.*)$", ""),
    ]
    for pat, repl in replacements:
        text = re.sub(pat, repl, text)

    text = re.sub(r"\s+", " ", text).strip(" -|_")
    return text

def deduplicate_segments(segments):
    deduped = []
    seen = set()
    for seg in segments:
        key = normalize_segment_for_dedup(seg)
        if not key:
            continue
        if key in seen:
            continue
        seen.add(key)
        deduped.append(seg.strip())
    return deduped

def normalize_display_title(text):
    text = safe_str(text)
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text).strip(" -|_")
    text = re.sub(r"\s*([/,+])\s*", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # mild title case only for obvious lowercase english tokens
    text = text.replace("BackEend", "Backend")
    text = text.replace("Devops", "DevOps")
    text = text.replace("FullStack", "Fullstack")
    text = text.replace("Full Stack", "Fullstack")
    text = text.replace("FrontEnd", "Frontend")
    text = text.replace("BackEnd", "Backend")
    text = text.replace("Ai ", "AI ")
    text = text.replace(" Ai", " AI")
    text = text.replace("DotNet", ".NET")
    text = text.replace(". Net", ".NET")
    text = text.replace(". net", ".NET")
    text = text.replace("Reactjs", "ReactJS")
    text = text.replace("Vuejs", "VueJS")
    text = text.replace("Nodejs", "NodeJS")
    text = text.replace("Golang", "Golang")
    text = text.replace("Npl", "NLP")

    for pat, repl in DISPLAY_REPLACEMENTS:
        text = re.sub(pat, repl, text)

    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\s+", " ", text).strip(" -|_")
    return text

def normalize_title_surface(text):
    text = clean_text_light(text)
    if not text:
        return ""

    text = clean_title_global_noise(text)
    text = strip_bracket_noise(text)
    text = clean_bracket_content(text)
    text = remove_internal_title_codes(text)

    segments = split_title_segments(text)
    segments = [seg.strip() for seg in segments if seg.strip()]
    kept_segments = [seg for seg in segments if not is_noise_segment(seg)]

    if kept_segments:
        kept_segments = deduplicate_segments(kept_segments)
        text = " - ".join(kept_segments)
    else:
        text = text.strip()

    # remove repeated canonical phrases joined by hyphen
    text = re.sub(r"\s+", " ", text).strip(" -|_")
    text = normalize_display_title(text)
    return text

def canonicalize_title(surface_text):
    s = normalize_for_match(surface_text)
    if not s:
        return ""

    s = re.sub(r"\b(junior|middle|senior|lead|leader|team lead|fresher|intern)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # direct exact lookup
    if s in TITLE_SYNONYM_MAP:
        return TITLE_SYNONYM_MAP[s]

    # exact phrase containment prioritizing longest synonym key
    for k in sorted(TITLE_SYNONYM_MAP.keys(), key=len, reverse=True):
        if k == s:
            return TITLE_SYNONYM_MAP[k]

    for k in sorted(TITLE_SYNONYM_MAP.keys(), key=len, reverse=True):
        if k in s:
            return TITLE_SYNONYM_MAP[k]

    # fallback by broader canonical keyword rules
    for canonical, keywords in TITLE_CANONICAL_KEYWORDS.items():
        for kw in keywords:
            if kw in s:
                return canonical

    # sensible defaults for uncaptured cases
    if "backend" in s or "java developer" in s or "python developer" in s or "nodejs" in s or "php developer" in s or ".net" in s or "dotnet" in s:
        return "backend developer"
    if "frontend" in s or "reactjs" in s or "angular" in s or "vuejs" in s or "react native" in s:
        return "frontend developer"
    if "fullstack" in s or "full stack" in s:
        return "fullstack developer"
    if "devops" in s or "devsecops" in s or "sre" in s:
        return "devops engineer"
    if "tester" in s or "kiem thu" in s or "qa" in s or "qc" in s:
        return "automation tester"
    if "business analyst" in s or re.search(r"\bba\b", s):
        return "business analyst"
    if "product owner" in s or "product manager" in s:
        return "product manager"
    if "data analyst" in s or "phan tich du lieu" in s:
        return "data analyst"
    if "data engineer" in s or "du lieu" in s and "engineer" in s:
        return "data engineer"
    if "data scientist" in s or "khoa hoc du lieu" in s:
        return "data scientist"
    if "ai" in s and ("engineer" in s or "developer" in s):
        return "ai engineer"
    if "research" in s and "ai" in s:
        return "ai researcher"
    if "cloud" in s:
        return "cloud engineer"
    if "dba" in s or "database administrator" in s or "quan tri co so du lieu" in s:
        return "database administrator"
    if "embedded" in s or "firmware" in s or "lap trinh nhung" in s:
        return "embedded engineer"
    if "iot" in s:
        return "iot engineer"
    if "label" in s or "gan nhan" in s or "xu ly du lieu" in s or "ngon ngu du lieu" in s:
        return "data labeling specialist"

    return s

def normalize_job_title(text):
    surface = normalize_title_surface(text)
    canonical = canonicalize_title(surface)
    return canonical

def infer_job_family_from_title(job_title_canonical):
    t = safe_str(job_title_canonical).lower().strip()
    if not t:
        return "unknown"

    for family, keywords in JOB_FAMILY_RULES.items():
        if any(k in t for k in keywords):
            return family
    return "unknown"

def infer_job_family_from_description(text):
    t = normalize_for_match(text)
    if not t:
        return "unknown"

    scores = {}
    for family, keywords in JOB_FAMILY_DESCRIPTION_HINTS.items():
        score = 0
        for kw in keywords:
            if kw in t:
                score += 1
        scores[family] = score

    best_family = max(scores, key=scores.get)
    best_score = scores[best_family]

    return best_family if best_score > 0 else "unknown"

def resolve_job_family(title_family, desc_family):
    title_family = safe_str(title_family) or "unknown"
    desc_family = safe_str(desc_family) or "unknown"

    if title_family != "unknown" and desc_family == "unknown":
        return title_family, "title"

    if title_family == "unknown" and desc_family != "unknown":
        return desc_family, "description"

    if title_family != "unknown" and desc_family != "unknown":
        if title_family == desc_family:
            return title_family, "title+description"
        return title_family, "title_priority"

    return "unknown", "unknown"


In [90]:
# =========================
# APPLY
# =========================

df_clean["job_title_surface"] = df_clean["job_title_raw"].apply(normalize_title_surface)
df_clean["job_title_match"] = df_clean["job_title_surface"].apply(normalize_for_match)
df_clean["job_title_canonical"] = df_clean["job_title_surface"].apply(canonicalize_title)

# giữ compatibility với code cũ nếu bạn còn dùng job_title_display ở chỗ khác
df_clean["job_title_display"] = df_clean["job_title_surface"].fillna("")

df_clean["job_family_from_title"] = df_clean["job_title_canonical"].apply(infer_job_family_from_title)
df_clean["job_family_from_description"] = df_clean["description_clean_strict"].apply(infer_job_family_from_description)

family_resolved = [
    resolve_job_family(a, b)
    for a, b in zip(df_clean["job_family_from_title"], df_clean["job_family_from_description"])
]
df_clean["job_family"] = [x[0] for x in family_resolved]
df_clean["job_family_source"] = [x[1] for x in family_resolved]

# PhoBERT: ưu tiên surface vì còn dấu + thêm canonical để neo nghĩa
df_clean["job_title_for_phobert"] = (
    df_clean["job_title_surface"].fillna("").str.strip()
    + " | "
    + df_clean["job_title_canonical"].fillna("").str.strip()
).str.strip(" |").apply(clean_text_for_phobert)

# Preview nhanh
preview_title_cols = [
    "job_title_raw",
    "job_title_surface",
    "job_title_match",
    "job_title_canonical",
    "job_family_from_title",
    "job_family_from_description",
    "job_family",
    "job_family_source",
    "job_title_for_phobert",
]
display(df_clean[preview_title_cols].head(10))

,job_title_raw,job_title_surface,job_title_match,job_title_canonical,job_family_from_title,job_family_from_description,job_family,job_family_source,job_title_for_phobert
0,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,Nhân Viên Tester Mobile App,nhan vien tester mobile app,automation tester,qa_testing,qa_testing,qa_testing,title+description,Nhân Viên Tester Mobile App automation tester
1,WordPress Developer - Senior Level - (20h - 05h) Remote,WordPress Developer - Senior Level,wordpress developer - senior level,wordpress developer - level,unknown,software_engineering,software_engineering,description,WordPress Developer - Senior Level wordpress developer - level
2,Data Processing Analyst,Data Processing Analyst,data processing analyst,data labeling specialist,data_governance_quality,software_engineering,data_governance_quality,title_priority,Data Processing Analyst data labeling specialist
3,Fullstack Developer ( NodeJs ) - Yêu Cầu Kinh Nghiệm Từ 4 Năm Trở Lên (CoolMate),Fullstack Developer (NodeJs),fullstack developer nodejs,fullstack developer,software_engineering,unknown,software_engineering,title,Fullstack Developer (NodeJs) fullstack developer
4,Nhân Viên Backend Developer - Remote,Nhân Viên Backend Developer,nhan vien backend developer,backend developer,software_engineering,software_engineering,software_engineering,title+description,Nhân Viên Backend Developer backend developer
5,AI Engineer - Sản Phẩm AI Edtech,AI Engineer - Sản Phẩm AI Edtech,ai engineer - san pham ai edtech,ai engineer,data_science_ml,data_science_ml,data_science_ml,title+description,AI Engineer - Sản Phẩm AI Edtech ai engineer
6,Chuyên Viên Phân Tích Nghiệp Vụ ( Business Analyst ),Chuyên Viên Phân Tích Nghiệp Vụ (Business Analyst),chuyen vien phan tich nghiep vu business analyst,business analyst,product_project_ba,unknown,product_project_ba,title,Chuyên Viên Phân Tích Nghiệp Vụ (Business Analyst) business analyst
7,Data Governance Specialist,Data Governance Specialist,data governance specialist,data governance specialist,unknown,data_governance_quality,data_governance_quality,description,Data Governance Specialist data governance specialist
8,Middle Java Developer,Middle Java Developer,middle java developer,backend developer,software_engineering,software_engineering,software_engineering,title+description,Middle Java Developer backend developer
9,Nhân Viên Kiểm Thử Phần Mềm,Nhân Viên Kiểm Thử Phần Mềm,nhan vien kiem thu phan mem,automation tester,qa_testing,unknown,qa_testing,title,Nhân Viên Kiểm Thử Phần Mềm automation tester


### - Chuẩn hóa địa điểm

In [91]:
# =========================
# LOCATION
# =========================

VIETNAM_CITY_ALIASES = {
    "cao bằng": "Cao Bằng",
    "cao bang": "Cao Bằng",
    "sơn la": "Sơn La",
    "son la": "Sơn La",
    "lai châu": "Lai Châu",
    "lai chau": "Lai Châu",
    "lạng sơn": "Lạng Sơn",
    "lang son": "Lạng Sơn",
    "tuyên quang": "Tuyên Quang",
    "tuyen quang": "Tuyên Quang",
    "lào cai": "Lào Cai",
    "lao cai": "Lào Cai",
    "thái nguyên": "Thái Nguyên",
    "thai nguyen": "Thái Nguyên",
    "điện biên": "Điện Biên",
    "dien bien": "Điện Biên",
    "phú thọ": "Phú Thọ",
    "phu tho": "Phú Thọ",
    "bắc ninh": "Bắc Ninh",
    "bac ninh": "Bắc Ninh",
    "hà nội": "TP. Hà Nội",
    "ha noi": "TP. Hà Nội",
    "quảng ninh": "Quảng Ninh",
    "quang ninh": "Quảng Ninh",
    "hải phòng": "TP. Hải Phòng",
    "hai phong": "TP. Hải Phòng",
    "hưng yên": "Hưng Yên",
    "hung yen": "Hưng Yên",
    "ninh bình": "Ninh Bình",
    "ninh binh": "Ninh Bình",
    "thanh hóa": "Thanh Hóa",
    "thanh hoa": "Thanh Hóa",
    "nghệ an": "Nghệ An",
    "nghe an": "Nghệ An",
    "hà tĩnh": "Hà Tĩnh",
    "ha tinh": "Hà Tĩnh",
    "quảng trị": "Quảng Trị",
    "quang tri": "Quảng Trị",
    "huế": "TP. Huế",
    "hue": "TP. Huế",
    "đà nẵng": "TP. Đà Nẵng",
    "da nang": "TP. Đà Nẵng",
    "quảng ngãi": "Quảng Ngãi",
    "quang ngai": "Quảng Ngãi",
    "gia lai": "Gia Lai",
    "đắk lắk": "Đắk Lắk",
    "dak lak": "Đắk Lắk",
    "khánh hòa": "Khánh Hòa",
    "khanh hoa": "Khánh Hòa",
    "lâm đồng": "Lâm Đồng",
    "lam dong": "Lâm Đồng",
    "đồng nai": "Đồng Nai",
    "dong nai": "Đồng Nai",
    "hồ chí minh": "TP. Hồ Chí Minh",
    "ho chi minh": "TP. Hồ Chí Minh",
    "tp hcm": "TP. Hồ Chí Minh",
    "tphcm": "TP. Hồ Chí Minh",
    "hcm": "TP. Hồ Chí Minh",
    "sài gòn": "TP. Hồ Chí Minh",
    "sai gon": "TP. Hồ Chí Minh",
    "tây ninh": "Tây Ninh",
    "tay ninh": "Tây Ninh",
    "đồng tháp": "Đồng Tháp",
    "dong thap": "Đồng Tháp",
    "vĩnh long": "Vĩnh Long",
    "vinh long": "Vĩnh Long",
    "cần thơ": "TP. Cần Thơ",
    "can tho": "TP. Cần Thơ",
    "an giang": "An Giang",
    "cà mau": "Cà Mau",
    "ca mau": "Cà Mau",
}

CITY_LABEL_PATTERN = r"(Cao Bằng|Cao Bang|Sơn La|Son La|Lai Châu|Lai Chau|Lạng Sơn|Lang Son|Tuyên Quang|Tuyen Quang|Lào Cai|Lao Cai|Thái Nguyên|Thai Nguyen|Điện Biên|Dien Bien|Phú Thọ|Phu Tho|Bắc Ninh|Bac Ninh|Hà Nội|Ha Noi|Quảng Ninh|Quang Ninh|Hải Phòng|Hai Phong|Hưng Yên|Hung Yen|Ninh Bình|Ninh Binh|Thanh Hóa|Thanh Hoa|Nghệ An|Nghe An|Hà Tĩnh|Ha Tinh|Quảng Trị|Quang Tri|Huế|Hue|Đà Nẵng|Da Nang|Quảng Ngãi|Quang Ngai|Gia Lai|Đắk Lắk|Dak Lak|Khánh Hòa|Khanh Hoa|Lâm Đồng|Lam Dong|Đồng Nai|Dong Nai|Hồ Chí Minh|Ho Chi Minh|TP\.?\s*HCM|TPHCM|HCM|Sài Gòn|Sai Gon|Tây Ninh|Tay Ninh|Đồng Tháp|Dong Thap|Vĩnh Long|Vinh Long|Cần Thơ|Can Tho|An Giang|Cà Mau|Ca Mau)\s*:"

def detect_city_from_text(text):
    t = normalize_for_match(text)
    hits = []

    for alias, canonical in VIETNAM_CITY_ALIASES.items():
        alias_norm = normalize_for_match(alias)
        if alias_norm and re.search(rf"(?<!\w){re.escape(alias_norm)}(?!\w)", t):
            hits.append(canonical)

    hits = list(dict.fromkeys(hits))
    return ", ".join(hits) if hits else None

def has_multi_location(text):
    t = normalize_for_match(text)
    hits = set()

    for alias, canonical in VIETNAM_CITY_ALIASES.items():
        alias_norm = normalize_for_match(alias)
        if alias_norm and re.search(rf"(?<!\w){re.escape(alias_norm)}(?!\w)", t):
            hits.add(canonical)

    return len(hits) >= 2

def clean_working_address_text(raw_text):
    text = safe_str(raw_text)
    if not text:
        return ""

    text = text.replace("\\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    # bỏ note editorial đầu dòng
    text = re.sub(
        r"^\(\s*đã được cập nhật theo danh mục hành chính mới.*?\)\s*[-:–•]*\s*",
        "",
        text,
        flags=re.I
    )

    # bỏ chú thích đơn vị hành chính cũ
    text = re.sub(
        r"\(\s*(quận|huyện|thị xã|thành phố|tp\.?)\s+[^)]*?\s+cũ\s*\)",
        "",
        text,
        flags=re.I
    )

    # bỏ "(Tất cả phường)"
    text = re.sub(
        r"\(\s*tất cả phường\s*\)",
        "",
        text,
        flags=re.I
    )

    # bỏ "...và N địa điểm khác Thu gọn"
    text = re.sub(
        r"\.\.\.\s*và\s*\d+\s*địa điểm khác\s*thu gọn",
        "",
        text,
        flags=re.I
    )

    # bỏ chuỗi "Thu gọn" còn sót
    text = re.sub(r"\bThu gọn\b", "", text, flags=re.I)

    # chuẩn bullet / dash đầu dòng
    text = re.sub(r"^\s*[-–•]+\s*", "", text)

    # chuẩn khoảng trắng
    text = re.sub(r"\s+", " ", text).strip(" -,:;")
    return text

def split_multiple_addresses(text):
    text = safe_str(text)
    if not text:
        return []

    matches = list(re.finditer(CITY_LABEL_PATTERN, text, flags=re.I))
    if not matches:
        return [text]

    parts = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip(" -|,;")
        if chunk:
            parts.append(chunk)

    return parts

def normalize_single_address_part(part):
    part = safe_str(part)
    if not part:
        return ""

    # bỏ lặp city label trong cùng 1 part nếu có
    part = re.sub(r"\s+", " ", part).strip(" -,:;")
    part = re.sub(r"\s*-\s*", " - ", part)
    part = re.sub(r"\s*,\s*", ", ", part)

    # bỏ các cụm vô nghĩa còn sót
    part = re.sub(r"\(\s*tất cả phường\s*\)", "", part, flags=re.I)
    part = re.sub(r"\bThu gọn\b", "", part, flags=re.I)

    # chuẩn lại khoảng trắng
    part = re.sub(r"\s+", " ", part).strip(" -,:;")
    return part

def is_meaningful_address_part(part):
    p = normalize_for_match(part)
    if not p:
        return False

    # loại part quá rỗng / vô nghĩa
    if p in {"tat ca phuong"}:
        return False

    # loại part chỉ còn city label
    city_only_patterns = [
        r"^(cao bang|son la|lai chau|lang son|tuyen quang|lao cai|thai nguyen|dien bien|phu tho|bac ninh|ha noi|quang ninh|hai phong|hung yen|ninh binh|thanh hoa|nghe an|ha tinh|quang tri|hue|da nang|quang ngai|gia lai|dak lak|khanh hoa|lam dong|dong nai|ho chi minh|tp hcm|tphcm|hcm|sai gon|tay ninh|dong thap|vinh long|can tho|an giang|ca mau)\s*:?$"
    ]
    if any(re.search(pat, p) for pat in city_only_patterns):
        return False

    return True

def deduplicate_preserve_order(items):
    seen = set()
    results = []
    for item in items:
        key = normalize_for_match(item)
        if not key or key in seen:
            continue
        seen.add(key)
        results.append(item)
    return results

def parse_working_address(raw_text):
    cleaned = clean_working_address_text(raw_text)
    parts = split_multiple_addresses(cleaned)
    parts = [normalize_single_address_part(p) for p in parts]
    parts = [p for p in parts if is_meaningful_address_part(p)]
    parts = deduplicate_preserve_order(parts)

    location_city = detect_city_from_text(raw_text)

    return {
        "working_address_clean_list": parts,
        "working_address_clean": " | ".join(parts) if parts else "",
        "location_city": location_city,
        "is_multi_location": len(parts) >= 2 or has_multi_location(raw_text),
    }

In [92]:
address_parsed = df_clean["working_addresses_raw"].apply(parse_working_address)
address_df = pd.DataFrame(address_parsed.tolist(), index=df_clean.index)

for c in address_df.columns:
    df_clean[c] = address_df[c]

In [93]:
display(df_clean[
    [
        "location_raw", "working_addresses_raw", "working_address_clean", "working_address_clean_list", "location_city", 
        "is_multi_location",
    ]
].head(5))

,location_raw,working_addresses_raw,working_address_clean,working_address_clean_list,location_city,is_multi_location
0,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân (quận Thanh Xuân cũ)","Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân","[Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân]",TP. Hà Nội,False
1,Hà Nội và 3 nơi khác,(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: (Tất cả phường) - Hồ Chí Minh: (Tất cả phường) - Đà Nẵng: (Tất cả phường) - Đồng Nai: (...,Đà Nẵng: - Đồng Nai,[Đà Nẵng: - Đồng Nai],"TP. Hà Nội, TP. Đà Nẵng, Đồng Nai, TP. Hồ Chí Minh",True
2,Hồ Chí Minh (mới),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:","Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận Hồ Chí Minh","[Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận Hồ Chí Minh]",TP. Hồ Chí Minh,False
3,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Toà BMM, Phường Hà Đông (quận Hà Đông cũ)","Hà Nội: Toà BMM, Phường Hà Đông","[Hà Nội: Toà BMM, Phường Hà Đông]",TP. Hà Nội,False
4,"Hồ Chí Minh , Hà Nội","(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: 115 Lý Chính Thắng, Phường Xuân Hòa (Quận 3 cũ) - Hà Nội: Tầng 5, Tòa nhà Artemis,...","Hồ Chí Minh: 115 Lý Chính Thắng, Phường Xuân Hòa | Hà Nội: Tầng 5, Tòa nhà Artemis, số 3 Lê Trọng Tấn, phường Phương Liệt, Phường Thanh Xuân","[Hồ Chí Minh: 115 Lý Chính Thắng, Phường Xuân Hòa, Hà Nội: Tầng 5, Tòa nhà Artemis, số 3 Lê Trọng Tấn, phường Phương Liệt, Phường Thanh Xuân]","TP. Hà Nội, TP. Hồ Chí Minh",True


### - Trích xuất word_mode

In [94]:
# =========================
# WORK MODE
# =========================

WORK_MODE_RULES = {
    "hybrid": [
        r"\bhybrid\b",
        r"\bhybird\b",
        r"\bhybrid working\b",
        r"\blinh hoat\b",
        r"\bket hop onsite va remote\b",
        r"\bket hop online va offline\b",
        r"\bket hop truc tiep va tu xa\b",
        r"\bremote \d+ days/month\b",
        r"\bremote \d+ ngay/tu[a-z]*\b",
        r"\b[0-9]+\s*days/month\b",
        r"\b[0-9]+\s*ngay/tu[a-z]*\b",
    ],
    "remote": [
        r"\bremote\b",
        r"\bwfh\b",
        r"\bwork from home\b",
        r"\bfully remote\b",
        r"\blam viec tu xa\b",
        r"\blam viec online\b",
        r"\blam online\b",
        r"\blam viec tai nha\b",
        r"\btai nha\b",
    ],
    "onsite": [
        r"\bonsite\b",
        r"\bon site\b",
        r"\bwork onsite\b",
        r"\bwork on site\b",
        r"\bin office\b",
        r"\btai van phong\b",
        r"\blam viec tai cong ty\b",
        r"\blam viec truc tiep\b",
        r"\blam viec co dinh tai vp\b",
    ],
}

def infer_work_mode(*texts):
    merged = " ".join([safe_str(t) for t in texts if safe_str(t)])
    merged_norm = normalize_for_match(merged)

    if not merged_norm:
        return "unknown"

    for mode in ["hybrid", "remote", "onsite"]:
        for p in WORK_MODE_RULES[mode]:
            if re.search(p, merged_norm, flags=re.I):
                return mode

    return "unknown"

def infer_work_mode_with_source(*texts):
    merged = " ".join([safe_str(t) for t in texts if safe_str(t)])
    merged_norm = normalize_for_match(merged)

    if not merged_norm:
        return "unknown", "none"

    for mode in ["hybrid", "remote", "onsite"]:
        for p in WORK_MODE_RULES[mode]:
            if re.search(p, merged_norm, flags=re.I):
                return mode, p

    return "unknown", "none"

In [95]:
work_mode_parsed = [
    infer_work_mode_with_source(a, b, c, d)
    for a, b, c, d in zip(
        df_clean["job_title_raw"],
        df_clean["description_raw"],
        df_clean["benefits_raw"],
        df_clean["requirements_raw"]
    )
]

df_clean["work_mode"] = [x[0] for x in work_mode_parsed]
df_clean["work_mode_source"] = [x[1] for x in work_mode_parsed]

In [96]:
df_clean["work_mode"].value_counts().head(5)

work_mode
unknown    1023
hybrid      206
onsite      190
remote       58
Name: count, dtype: int64

### - Chuẩn hóa deadline

In [97]:
# =========================
# DEADLINE
# =========================

def parse_deadline(raw, reference_date=None):
    reference_date = reference_date or datetime.today().date()
    text = clean_text_light(raw)

    if not text:
        return {
            "deadline_clean": "",
            "deadline_date": None,
            "days_to_deadline": None,
            "deadline_type": "unknown",
            "is_expired": None,
        }

    m = re.search(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", text)
    if m:
        day, month, year = map(int, m.groups())
        if year < 100:
            year += 2000
        try:
            dt = datetime(year, month, day).date()
            return {
                "deadline_clean": text,
                "deadline_date": str(dt),
                "days_to_deadline": (dt - reference_date).days,
                "deadline_type": "absolute_date",
                "is_expired": dt < reference_date,
            }
        except Exception:
            pass

    m = re.search(r"(\d+)\s*ngay", normalize_for_match(text))
    if m:
        days = int(m.group(1))
        dt = reference_date + timedelta(days=days)
        return {
            "deadline_clean": text,
            "deadline_date": str(dt),
            "days_to_deadline": days,
            "deadline_type": "relative_days",
            "is_expired": False,
        }

    return {
        "deadline_clean": text,
        "deadline_date": None,
        "days_to_deadline": None,
        "deadline_type": "unknown",
        "is_expired": None,
    }

deadline_parsed = df_clean["deadline_raw"].apply(parse_deadline)
deadline_df = pd.DataFrame(deadline_parsed.tolist(), index=df_clean.index)

for c in deadline_df.columns:
    df_clean[c] = deadline_df[c]

In [98]:
display(df_clean[
    [
        "deadline_raw","deadline_clean", "deadline_date", "days_to_deadline", "deadline_type", "is_expired"
    ]
].head(10))

,deadline_raw,deadline_clean,deadline_date,days_to_deadline,deadline_type,is_expired
0,13/04/2026,13/04/2026,2026-04-13,11.0,absolute_date,False
1,05/04/2026,05/04/2026,2026-04-05,3.0,absolute_date,False
2,Còn 29 ngày để ứng tuyển,Còn 29 ngày để ứng tuyển,2026-05-01,29.0,relative_days,False
3,15/04/2026,15/04/2026,2026-04-15,13.0,absolute_date,False
4,26/04/2026,26/04/2026,2026-04-26,24.0,absolute_date,False
5,Còn 26 ngày để ứng tuyển,Còn 26 ngày để ứng tuyển,2026-04-28,26.0,relative_days,False
6,Còn 28 ngày để ứng tuyển,Còn 28 ngày để ứng tuyển,2026-04-30,28.0,relative_days,False
7,Còn 23 ngày để ứng tuyển,Còn 23 ngày để ứng tuyển,2026-04-25,23.0,relative_days,False
8,Còn 29 ngày để ứng tuyển,Còn 29 ngày để ứng tuyển,2026-05-01,29.0,relative_days,False
9,Còn 27 ngày để ứng tuyển,Còn 27 ngày để ứng tuyển,2026-04-29,27.0,relative_days,False


### - Chuẩn hóa lương

In [99]:
# =========================
# SALARY
# =========================

def clean_salary(raw):
    text = clean_text_light(raw).lower()
    text = text.replace("thoả", "thỏa")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_numeric_token(num_text):
    s = safe_str(num_text).strip().replace(" ", "")
    if not s:
        return None

    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    elif s.count(",") > 1:
        s = s.replace(",", "")
    else:
        s = s.replace(",", ".")

    try:
        return float(s)
    except Exception:
        return None


def detect_salary_multiplier(text):
    t = safe_str(text).lower()

    # USD giữ nguyên base number, sẽ quy đổi sau
    if "usd" in t or "$" in t:
        return 1.0, "usd"

    if "triệu" in t or "trieu" in t or re.search(r"\b\d+(?:[.,]\d+)?\s*(tr|trieu)\b", t):
        return 1_000_000, "vnd"

    if "nghìn" in t or "ngan" in t or re.search(r"\b\d+(?:[.,]\d+)?\s*k\b", t):
        return 1_000, "vnd"

    return 1.0, "vnd"


def make_salary_bucket(min_vnd, max_vnd, salary_type, negotiable):
    if negotiable:
        return "negotiable"

    ref = max_vnd if max_vnd is not None else min_vnd
    if ref is None:
        return "unknown"

    if ref < 10_000_000:
        return "under_10m"
    if ref < 20_000_000:
        return "10m_20m"
    if ref < 30_000_000:
        return "20m_30m"
    if ref < 50_000_000:
        return "30m_50m"
    return "50m_plus"


def parse_salary_range(raw):
    text = clean_salary(raw)
    if not text:
        return {
            "salary_clean": "",
            "salary_type": "unknown",
            "salary_is_negotiable": None,
            "salary_min_vnd_month": None,
            "salary_max_vnd_month": None,
            "salary_bucket": "unknown",
        }

    negotiable = ("thỏa thuận" in text) or ("negotiable" in text)

    nums = re.findall(r"\d+(?:[.,]\d+)?", text)
    nums = [parse_numeric_token(x) for x in nums]
    nums = [x for x in nums if x is not None]

    multiplier, currency = detect_salary_multiplier(text)

    salary_min = None
    salary_max = None
    salary_type = "unknown"

    if negotiable and not nums:
        salary_type = "negotiable"

    elif "up to" in text or "upto" in text or "tối đa" in text:
        if nums:
            salary_max = nums[0] * multiplier
            salary_type = "upper_bound"

    elif len(nums) >= 2:
        salary_min = nums[0] * multiplier
        salary_max = nums[1] * multiplier
        salary_type = "range"

    elif len(nums) == 1:
        salary_min = nums[0] * multiplier
        salary_max = nums[0] * multiplier
        salary_type = "fixed"

    # Quy đổi toàn bộ về VND/tháng để filter thống nhất
    if currency == "usd":
        fx = 25000
        if salary_min is not None:
            salary_min *= fx
        if salary_max is not None:
            salary_max *= fx

    salary_bucket = make_salary_bucket(
        salary_min,
        salary_max,
        salary_type,
        negotiable
    )

    return {
        "salary_clean": text,
        "salary_type": salary_type,
        "salary_is_negotiable": negotiable,
        "salary_min_vnd_month": salary_min,
        "salary_max_vnd_month": salary_max,
        "salary_bucket": salary_bucket,
    }

In [100]:

salary_parsed = df_clean["salary_raw"].apply(parse_salary_range)
salary_df = pd.DataFrame(salary_parsed.tolist(), index=df_clean.index)

for c in salary_df.columns:
    df_clean[c] = salary_df[c]

display(df_clean[
    [
        "salary_raw",
        "salary_clean",
        "salary_type",
        "salary_is_negotiable",
        "salary_min_vnd_month",
        "salary_max_vnd_month",
        "salary_bucket",
    ]
].head(10))

,salary_raw,salary_clean,salary_type,salary_is_negotiable,salary_min_vnd_month,salary_max_vnd_month,salary_bucket
0,Thoả thuận,thỏa thuận,negotiable,True,NaN,NaN,negotiable
1,12 - 18 triệu,12 - 18 triệu,range,False,12000000.0,18000000.0,10m_20m
2,25 - 30 triệu,25 - 30 triệu,range,False,25000000.0,30000000.0,30m_50m
3,20 - 25 triệu,20 - 25 triệu,range,False,20000000.0,25000000.0,20m_30m
4,20 - 30 triệu,20 - 30 triệu,range,False,20000000.0,30000000.0,30m_50m
5,30 - 40 triệu,30 - 40 triệu,range,False,30000000.0,40000000.0,30m_50m
6,Tới 18 triệu,tới 18 triệu,fixed,False,18000000.0,18000000.0,10m_20m
7,20 - 30 triệu,20 - 30 triệu,range,False,20000000.0,30000000.0,30m_50m
8,Tới 25 triệu,tới 25 triệu,fixed,False,25000000.0,25000000.0,20m_30m
9,Tới 15 triệu,tới 15 triệu,fixed,False,15000000.0,15000000.0,10m_20m


### - Chuẩn hóa exp, education_level, employ_type, job_level, 

In [101]:
# =========================
# EXPERIENCE + EDUCATION + EMPLOYMENT TYPE + JOB LEVEL
# =========================

def clean_experience(text):
    return normalize_for_match(text)


def _to_years(value, unit):
    if value is None:
        return None
    unit = safe_str(unit).lower()
    if "thang" in unit:
        return value / 12.0
    return value


def parse_experience_range(raw):
    text = clean_experience(raw)
    if not text:
        return {
            "experience_clean": "",
            "experience_min_years": None,
            "experience_max_years": None,
            "experience_type": "unknown",
        }

    # no experience
    if any(kw in text for kw in [
        "khong yeu cau kinh nghiem",
        "khong can kinh nghiem",
        "khong yeu cau",
        "no experience",
        "khong co kinh nghiem",
    ]):
        return {
            "experience_clean": text,
            "experience_min_years": 0.0,
            "experience_max_years": 0.0,
            "experience_type": "no_experience",
        }

    # range with explicit units, e.g. "6 thang - 1 nam", "tu 2 den 3 nam"
    m = re.search(
        r"(\d+(?:\.\d+)?)\s*(thang|nam)\s*(?:-|den|toi)\s*(\d+(?:\.\d+)?)\s*(thang|nam)",
        text
    )
    if m:
        v1 = float(m.group(1))
        u1 = m.group(2)
        v2 = float(m.group(3))
        u2 = m.group(4)
        y1 = _to_years(v1, u1)
        y2 = _to_years(v2, u2)
        return {
            "experience_clean": text,
            "experience_min_years": min(y1, y2),
            "experience_max_years": max(y1, y2),
            "experience_type": "range",
        }

    # minimum explicit unit: "tu 6 thang", "tren 2 nam", "it nhat 1 nam", "2+ nam"
    m = re.search(
        r"(?:tu|tren|hon|it nhat|toi thieu)?\s*(\d+(?:\.\d+)?)\s*(\+)?\s*(thang|nam)",
        text
    )
    if m:
        val = float(m.group(1))
        plus = m.group(2)
        unit = m.group(3)
        years = _to_years(val, unit)

        if any(x in text for x in ["tu ", "tren ", "hon ", "it nhat", "toi thieu"]) or plus:
            return {
                "experience_clean": text,
                "experience_min_years": years,
                "experience_max_years": None,
                "experience_type": "minimum",
            }

        if "duoi" in text:
            return {
                "experience_clean": text,
                "experience_min_years": 0.0,
                "experience_max_years": years,
                "experience_type": "maximum",
            }

        return {
            "experience_clean": text,
            "experience_min_years": years,
            "experience_max_years": years,
            "experience_type": "fixed",
        }

    # range without repeated unit at end: "2-3 nam", "2 den 3 nam"
    m = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:-|den|toi)\s*(\d+(?:\.\d+)?)\s*(thang|nam)",
        text
    )
    if m:
        v1 = float(m.group(1))
        v2 = float(m.group(2))
        unit = m.group(3)
        y1 = _to_years(v1, unit)
        y2 = _to_years(v2, unit)
        return {
            "experience_clean": text,
            "experience_min_years": min(y1, y2),
            "experience_max_years": max(y1, y2),
            "experience_type": "range",
        }

    # fallback numeric parse
    nums = re.findall(r"\d+(?:\.\d+)?", text)
    nums = [float(x) for x in nums] if nums else []

    if len(nums) >= 2:
        return {
            "experience_clean": text,
            "experience_min_years": min(nums[0], nums[1]),
            "experience_max_years": max(nums[0], nums[1]),
            "experience_type": "range",
        }

    if len(nums) == 1:
        val = nums[0]
        if any(x in text for x in ["tu", "+", "tren", "it nhat", "toi thieu", "tro len"]):
            return {
                "experience_clean": text,
                "experience_min_years": val,
                "experience_max_years": None,
                "experience_type": "minimum",
            }
        if "duoi" in text:
            return {
                "experience_clean": text,
                "experience_min_years": 0.0,
                "experience_max_years": val,
                "experience_type": "maximum",
            }
        return {
            "experience_clean": text,
            "experience_min_years": val,
            "experience_max_years": val,
            "experience_type": "fixed",
        }

    return {
        "experience_clean": text,
        "experience_min_years": None,
        "experience_max_years": None,
        "experience_type": "unknown",
    }


EDUCATION_MAP = {
    "phd": ["tien si", "phd", "doctor", "doctoral"],
    "master": ["thac si", "master", "mba","cao hoc"],
    "bachelor": ["dai hoc", "cu nhan", "bachelor", "university"],
    "college": ["cao dang", "college","trung cap"],
    "high_school": ["trung hoc", "high school", "pho thong"],
}

EDUCATION_RANK = {
    "unknown": 0,
    "high_school": 1,
    "college": 2,
    "bachelor": 3,
    "master": 4,
    "phd": 5,
}

EMPLOYMENT_TYPE_MAP = {
    "full_time": [
        "toan thoi gian", "full time", "full-time", "fulltime", "chinh thuc"
    ],
    "part_time": [
        "ban thoi gian", "part time", "part-time", "parttime"
    ],
    "internship": [
        "thuc tap", "thuc tap sinh", "internship", "intern"
    ],
    "contract": [
        "hop dong", "contract"
    ],
    "freelance": [
        "freelance", "freelancer", "cong tac vien", "ctv", "cong tac"
    ],
    "temporary": [
        "temporary", "thoi vu", "ngan han"
    ],
}


def normalize_education_level(text):
    t = normalize_for_match(text)
    if not t:
        return "unknown"

    matched_levels = []
    for level, kws in EDUCATION_MAP.items():
        if any(k in t for k in kws):
            matched_levels.append(level)

    if not matched_levels:
        return "unknown"

    return max(matched_levels, key=lambda x: EDUCATION_RANK.get(x, 0))


def normalize_employment_type(text):
    t = normalize_for_match(text)
    if not t:
        return "unknown"

    for level, kws in EMPLOYMENT_TYPE_MAP.items():
        if any(k in t for k in kws):
            return level
    return "unknown"

# =========================
# JOB LEVEL (PRIORITY FILL)
# =========================

JOB_LEVEL_RULES = {
    "director": [
        r"\bdirector\b", r"\bhead\b", r"giam doc"
    ],
    "manager": [
        r"\bmanager\b", r"truong phong", r"pho phong", r"quan ly", r"giam sat"
    ],
    "lead": [
        r"\blead\b", r"\bleader\b", r"team lead", r"truong nhom"
    ],
    "senior": [
        r"\bsenior\b", r"\bsr\b", r"cao cap", r"chuyen vien cao cap"
    ],
    "middle": [
        r"\bmiddle\b", r"\bmid\b", r"mid senior", r"middle senior"
    ],
    "junior": [
        r"\bjunior\b", r"\bjr\b", r"nhan vien"
    ],
    "fresher": [
        r"\bfresher\b", r"moi tot nghiep"
    ],
    "intern": [
        r"\bintern\b", r"\binternship\b", r"thuc tap", r"thuc tap sinh"
    ],
}

JOB_LEVEL_PRIORITY = [
    "director", "manager", "lead", "senior", "middle", "junior", "fresher", "intern"
]

def extract_job_level_from_text(text):
    t = normalize_for_match(text)
    if not t:
        return "unknown"

    for lvl in JOB_LEVEL_PRIORITY:
        for p in JOB_LEVEL_RULES[lvl]:
            if re.search(p, t, flags=re.I):
                return lvl
    return "unknown"

def infer_job_level_from_experience(experience_min_years):
    if experience_min_years is None:
        return "unknown"

    if experience_min_years <= 0.5:
        return "fresher"
    if experience_min_years < 2.5:
        return "junior"
    if experience_min_years < 4:
        return "middle"
    return "senior"

def resolve_job_level(job_title_raw, experience_min_years, job_level_raw):
    # Priority 1: title
    lvl_from_title = extract_job_level_from_text(job_title_raw)
    if lvl_from_title != "unknown":
        return lvl_from_title, "job_title"

    # Priority 2: experience
    lvl_from_exp = infer_job_level_from_experience(experience_min_years)
    if lvl_from_exp != "unknown":
        return lvl_from_exp, "experience_min_years"

    # Priority 3: job_level_raw
    lvl_from_raw = extract_job_level_from_text(job_level_raw)
    if lvl_from_raw != "unknown":
        return lvl_from_raw, "job_level_raw"

    return "unknown", "unknown"

In [102]:
exp_parsed = df_clean["experience_raw"].apply(parse_experience_range)
exp_df = pd.DataFrame(exp_parsed.tolist(), index=df_clean.index)
for c in exp_df.columns:
    df_clean[c] = exp_df[c]

df_clean["education_level_norm"] = df_clean["education_level_raw"].apply(normalize_education_level)
df_clean["employment_type_norm"] = df_clean["employment_type_raw"].apply(normalize_employment_type)
job_level_parsed = [
    resolve_job_level(a, b, c)
    for a, b, c in zip(
        df_clean["job_title_raw"],
        df_clean["experience_min_years"],
        df_clean["job_level_raw"]
    )
]

df_clean["job_level_norm"] = [x[0] for x in job_level_parsed]
df_clean["job_level_source"] = [x[1] for x in job_level_parsed]

display(df_clean[
    [
        "experience_raw", "experience_clean", "experience_min_years", "experience_max_years", "experience_type",
        "education_level_raw", "education_level_norm",
        "employment_type_raw", "employment_type_norm",
        "job_level_raw", "job_title_raw", "job_level_norm"

    ]
].head(5))

,experience_raw,experience_clean,experience_min_years,experience_max_years,experience_type,education_level_raw,education_level_norm,employment_type_raw,employment_type_norm,job_level_raw,job_title_raw,job_level_norm
0,1 năm,1 nam,1.0,1.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,junior
1,3 năm,3 nam,3.0,3.0,fixed,Cao Đẳng trở lên,college,Toàn thời gian,full_time,Nhân viên,WordPress Developer - Senior Level - (20h - 05h) Remote,senior
2,2 năm,2 nam,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,Data Processing Analyst,junior
3,4 năm,4 nam,4.0,4.0,fixed,Cao Đẳng trở lên,college,Toàn thời gian,full_time,Nhân viên,Fullstack Developer ( NodeJs ) - Yêu Cầu Kinh Nghiệm Từ 4 Năm Trở Lên (CoolMate),senior
4,3 năm,3 nam,3.0,3.0,fixed,Cao Đẳng trở lên,college,Toàn thời gian,full_time,Nhân viên,Nhân Viên Backend Developer - Remote,junior


In [103]:
df_clean["job_level_norm"].value_counts(dropna=False)

job_level_norm
junior      612
middle      306
senior      211
fresher     102
lead         97
intern       90
manager      53
director      6
Name: count, dtype: int64

### - Chuẩn hóa tag

In [104]:
df_clean["tags_raw"].value_counts(dropna=False)

tags_raw
Backend Developer; IT - Phần mềm                                                                                                               85
Software Engineer; IT - Phần mềm                                                                                                               76
Business Analyst (Phân tích nghiệp vụ); IT - Phần mềm                                                                                          67
Software Tester (Automation & Manual); IT - Phần mềm                                                                                           45
Fullstack Developer; IT - Phần mềm                                                                                                             45
Business Analyst (Phân tích nghiệp vụ)                                                                                                         42
AI Engineer; IT - Phần mềm                                                                                         

In [105]:
# =========================
# TAGS
# =========================

TAG_ROLE_MAP = {
    "backend developer": "backend developer",
    "back end developer": "backend developer",
    "backend engineer": "backend developer",
    "software engineer": "software engineer",
    "software developer": "software engineer",
    "business analyst": "business analyst",
    "business analyst phan tich nghiep vu": "business analyst",
    "phan tich nghiep vu": "business analyst",
    "software tester": "software tester",
    "tester": "software tester",
    "qa tester": "software tester",
    "fullstack developer": "fullstack developer",
    "full stack developer": "fullstack developer",
    "frontend developer": "frontend developer",
    "front end developer": "frontend developer",
    "data analyst": "data analyst",
    "data engineer": "data engineer",
    "data scientist": "data scientist",
    "ai engineer": "ai engineer",
    "product owner": "product owner",
    "product manager": "product manager",
    "devops engineer": "devops engineer",
    "cloud engineer": "cloud engineer",
    "database administrator": "database administrator",
    "dba": "database administrator",
    "embedded engineer": "embedded engineer",
    "iot engineer": "iot engineer",
}

TAG_DOMAIN_KEYWORDS = {
    "it - phan mem": "it phan mem",
    "it phan mem": "it phan mem",
    "phan mem": "it phan mem",
    "software": "it phan mem",
    "it - phan cung": "it phan cung",
    "phan cung": "it phan cung",
    "du lieu": "du lieu",
    "data": "du lieu",
    "ai": "ai ml",
    "machine learning": "ai ml",
    "ml": "ai ml",
    "tai chinh": "tai chinh ngan hang",
    "ngan hang": "tai chinh ngan hang",
    "banking": "tai chinh ngan hang",
    "finance": "tai chinh ngan hang",
    "thuong mai dien tu": "thuong mai dien tu",
    "ecommerce": "thuong mai dien tu",
    "e commerce": "thuong mai dien tu",
    "edtech": "edtech",
    "fintech": "fintech",
    "healthcare": "healthcare",
    "y te": "healthcare",
    "game": "game",
}

TAG_SPECIALTY_KEYWORDS = {
    "automation": "automation",
    "manual": "manual",
    "api": "api",
    "mobile": "mobile",
    "web": "web",
    "backend": "backend",
    "frontend": "frontend",
    "fullstack": "fullstack",
    "full stack": "fullstack",
    "cloud": "cloud",
    "devops": "devops",
    "data": "data",
    "ai": "ai",
    "machine learning": "machine learning",
    "ml": "machine learning",
    "nlp": "nlp",
    "llm": "llm",
    "rag": "rag",
    "computer vision": "computer vision",
    "cv": "computer vision",
    "embedded": "embedded",
    "iot": "iot",
    "database": "database",
    "dba": "database",
    "product": "product",
    "business analysis": "business analysis",
    "phan tich nghiep vu": "business analysis",
}

def split_tag_parts(text):
    text = clean_text_light(text)
    if not text:
        return []
    return [p.strip() for p in re.split(r"[;\n]+", text) if p.strip()]

def normalize_single_tag(part):
    part = clean_text_light(part)
    if not part:
        return ""
    part = re.sub(r"\s+", " ", part).strip(" -,:;/|")
    return part

def extract_parenthetical_text(tag_text):
    matches = re.findall(r"\((.*?)\)", safe_str(tag_text))
    return [clean_text_light(m) for m in matches if clean_text_light(m)]

def strip_parenthetical_text(tag_text):
    t = safe_str(tag_text)
    t = re.sub(r"\(.*?\)", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return clean_text_light(t)

def extract_role_from_tag(tag_text):
    candidates = []

    core = strip_parenthetical_text(tag_text)
    if core:
        candidates.append(core)

    candidates.extend(extract_parenthetical_text(tag_text))

    for cand in candidates:
        c = normalize_for_match(cand)
        for k, v in sorted(TAG_ROLE_MAP.items(), key=lambda x: len(x[0]), reverse=True):
            if k in c:
                return v
    return None

def extract_domain_from_tag(tag_text):
    t = normalize_for_match(tag_text)

    for kw, canonical in sorted(TAG_DOMAIN_KEYWORDS.items(), key=lambda x: len(x[0]), reverse=True):
        if kw in t:
            return canonical
    return None

def extract_specialties_from_tag(tag_text):
    found = []

    base_text = normalize_for_match(tag_text)
    parenthetical_text = " ".join(normalize_for_match(x) for x in extract_parenthetical_text(tag_text))
    merged = f"{base_text} {parenthetical_text}".strip()

    for kw, canonical in sorted(TAG_SPECIALTY_KEYWORDS.items(), key=lambda x: len(x[0]), reverse=True):
        if kw in merged:
            found.append(canonical)

    return deduplicate_list(found)

def normalize_tags_structured(text):
    parts = split_tag_parts(text)
    parts_clean = [normalize_single_tag(p) for p in parts]
    parts_clean = [p for p in parts_clean if p]
    parts_clean = deduplicate_list(parts_clean)

    roles = []
    domains = []
    specialties = []

    for p in parts_clean:
        role = extract_role_from_tag(p)
        if role:
            roles.append(role)

        domain = extract_domain_from_tag(p)
        if domain:
            domains.append(domain)

        specialties.extend(extract_specialties_from_tag(p))

    roles = deduplicate_list(roles)
    domains = deduplicate_list(domains)
    specialties = deduplicate_list(specialties)

    # text cho model: giữ dấu tiếng Việt từ tag clean gốc
    tags_text_for_model = ", ".join(parts_clean)

    # text canonical cho rule/debug/scoring
    tags_text_canonical = ", ".join(deduplicate_list(roles + specialties + domains))

    return {
        "tags_list": parts_clean,
        "tags_role_list": roles,
        "tags_domain_list": domains,
        "tags_specialty_list": specialties,
        "tags_text_for_model": clean_text_for_phobert(tags_text_for_model),
        "tags_text_canonical": tags_text_canonical,
    }


In [106]:

tags_parsed = df_clean["tags_raw"].apply(normalize_tags_structured)
tags_df = pd.DataFrame(tags_parsed.tolist(), index=df_clean.index)

for c in tags_df.columns:
    df_clean[c] = tags_df[c]

display(df_clean[
    [
        "tags_raw",
        "tags_list",
        "tags_role_list",
        "tags_domain_list",
        "tags_specialty_list",
        "tags_text_for_model",
        "tags_text_canonical",
    ]
].head(10))

,tags_raw,tags_list,tags_role_list,tags_domain_list,tags_specialty_list,tags_text_for_model,tags_text_canonical
0,Chuyên môn Software Tester (Automation & Manual); IT - Phần cứng và máy tính; IT - Phần mềm; Thương mại điện tử,"[Chuyên môn Software Tester (Automation & Manual), IT - Phần cứng và máy tính, IT - Phần mềm, Thương mại điện tử]",[software tester],"[it phan mem, it phan cung, thuong mai dien tu]","[automation, manual, ai]","Chuyên môn Software Tester (Automation Manual), IT - Phần cứng và máy tính, IT - Phần mềm, Thương mại điện tử","software tester, automation, manual, ai, it phan mem, it phan cung, thuong mai dien tu"
1,Chuyên môn Frontend Developer; IT - Phần mềm,"[Chuyên môn Frontend Developer, IT - Phần mềm]",[frontend developer],[it phan mem],[frontend],"Chuyên môn Frontend Developer, IT - Phần mềm","frontend developer, frontend, it phan mem"
2,Chuyên môn Data Analyst,[Chuyên môn Data Analyst],[data analyst],[du lieu],[data],Chuyên môn Data Analyst,"data analyst, data, du lieu"
3,Chuyên môn Backend Developer; IT - Phần mềm; Thương mại điện tử; IT - Phần cứng và máy tính,"[Chuyên môn Backend Developer, IT - Phần mềm, Thương mại điện tử, IT - Phần cứng và máy tính]",[backend developer],"[it phan mem, thuong mai dien tu, it phan cung]","[backend, ai]","Chuyên môn Backend Developer, IT - Phần mềm, Thương mại điện tử, IT - Phần cứng và máy tính","backend developer, backend, ai, it phan mem, thuong mai dien tu, it phan cung"
4,Chuyên môn Backend Developer; IT - Phần mềm; Tài chính; Chứng khoán,"[Chuyên môn Backend Developer, IT - Phần mềm, Tài chính, Chứng khoán]",[backend developer],"[it phan mem, tai chinh ngan hang]","[backend, ai]","Chuyên môn Backend Developer, IT - Phần mềm, Tài chính, Chứng khoán","backend developer, backend, ai, it phan mem, tai chinh ngan hang"
5,Chuyên môn AI Engineer,[Chuyên môn AI Engineer],[ai engineer],[ai ml],[ai],Chuyên môn AI Engineer,"ai engineer, ai, ai ml"
6,Chuyên môn Business Analyst (Phân tích nghiệp vụ),[Chuyên môn Business Analyst (Phân tích nghiệp vụ)],[business analyst],[],[business analysis],Chuyên môn Business Analyst (Phân tích nghiệp vụ),"business analyst, business analysis"
7,Chuyên môn Data Analyst,[Chuyên môn Data Analyst],[data analyst],[du lieu],[data],Chuyên môn Data Analyst,"data analyst, data, du lieu"
8,Chuyên môn Backend Developer,[Chuyên môn Backend Developer],[backend developer],[],[backend],Chuyên môn Backend Developer,"backend developer, backend"
9,Chuyên môn Software Tester (Automation & Manual),[Chuyên môn Software Tester (Automation & Manual)],[software tester],[it phan mem],"[automation, manual]",Chuyên môn Software Tester (Automation Manual),"software tester, automation, manual, it phan mem"


### - SKILL TAXONOMY

In [107]:
# =========================
# SKILL TAXONOMY
# =========================

SKILL_TAXONOMY = {
    "python": {"aliases": ["python"], "group": "programming"},
    "sql": {"aliases": ["sql", "postgresql", "mysql", "sql server"], "group": "database"},
    "excel": {"aliases": ["excel", "microsoft excel"], "group": "analytics_tools"},
    "power bi": {"aliases": ["power bi", "powerbi"], "group": "bi_tools"},
    "tableau": {"aliases": ["tableau"], "group": "bi_tools"},
    "pandas": {"aliases": ["pandas"], "group": "python_libs"},
    "numpy": {"aliases": ["numpy"], "group": "python_libs"},
    "scikit-learn": {"aliases": ["scikit-learn", "sklearn"], "group": "ml_libs"},
    "tensorflow": {"aliases": ["tensorflow"], "group": "ml_libs"},
    "pytorch": {"aliases": ["pytorch", "torch"], "group": "ml_libs"},
    "machine learning": {"aliases": ["machine learning", "ml"], "group": "ml_concepts"},
    "deep learning": {"aliases": ["deep learning", "dl"], "group": "ml_concepts"},
    "statistics": {"aliases": ["statistics", "thống kê"], "group": "analytics_concepts"},
    "etl": {"aliases": ["etl"], "group": "data_engineering"},
    "airflow": {"aliases": ["airflow", "apache airflow"], "group": "data_engineering"},
    "spark": {"aliases": ["spark", "apache spark", "pyspark"], "group": "data_engineering"},
    "hadoop": {"aliases": ["hadoop"], "group": "data_engineering"},
    "kafka": {"aliases": ["kafka", "apache kafka"], "group": "data_engineering"},
    "aws": {"aliases": ["aws", "amazon web services"], "group": "cloud"},
    "gcp": {"aliases": ["gcp", "google cloud platform"], "group": "cloud"},
    "azure": {"aliases": ["azure"], "group": "cloud"},
    "docker": {"aliases": ["docker"], "group": "devops"},
    "kubernetes": {"aliases": ["kubernetes", "k8s"], "group": "devops"},
    "git": {"aliases": ["git", "github", "gitlab"], "group": "dev_tools"},
    "api": {"aliases": ["api", "rest api", "restful api"], "group": "backend"},
    "fastapi": {"aliases": ["fastapi"], "group": "backend"},
    "flask": {"aliases": ["flask"], "group": "backend"},
    "llm": {"aliases": ["llm", "large language model", "large language models"], "group": "ai"},
    "nlp": {"aliases": ["nlp", "natural language processing"], "group": "ai"},
    "computer vision": {"aliases": ["computer vision", "cv"], "group": "ai"},
    "data modeling": {"aliases": ["data modeling", "data model"], "group": "data_modeling"},
    "data warehouse": {"aliases": ["data warehouse", "dwh"], "group": "data_modeling"},
    "bigquery": {"aliases": ["bigquery"], "group": "database"},
    "snowflake": {"aliases": ["snowflake"], "group": "database"},
    "oracle": {"aliases": ["oracle"], "group": "database"},
    "sas": {"aliases": ["sas"], "group": "analytics_tools"},
    "r": {"aliases": ["r", "ngôn ngữ r"], "group": "programming"},
    "communication": {"aliases": ["communication", "giao tiếp"], "group": "soft_skills"},
    "problem solving": {"aliases": ["problem solving", "giải quyết vấn đề"], "group": "soft_skills"},
}

REQUIRED_HINTS = [
    "bắt buộc", "required", "must have", "cần có", "thành thạo", "proficient", "kinh nghiệm với"
]

PREFERRED_HINTS = [
    "ưu tiên", "preferred", "nice to have", "là lợi thế", "plus point"
]


In [108]:
def alias_to_regex(alias):
    alias = safe_str(alias).strip()
    if not alias:
        return None

    alias_norm = normalize_for_match(alias)

    # các alias quá ngắn cần boundary chặt hơn
    if alias_norm == "r":
        return r"(?<!\w)r(?!\w)"
    if alias_norm == "ml":
        return r"(?<!\w)ml(?!\w)"
    if alias_norm == "cv":
        return r"(?<!\w)cv(?!\w)"

    return r"(?<!\w)" + re.escape(alias_norm) + r"(?!\w)"


SKILL_PATTERNS = {}
for skill, meta in SKILL_TAXONOMY.items():
    pats = []
    for alias in meta["aliases"]:
        pat = alias_to_regex(alias)
        if pat:
            pats.append(re.compile(pat, flags=re.I))
    SKILL_PATTERNS[skill] = pats


def infer_skill_importance(segment, source_field):
    s = normalize_for_match(segment)

    if any(h in s for h in PREFERRED_HINTS):
        return "preferred"

    if source_field == "requirements":
        return "required"

    if any(h in s for h in REQUIRED_HINTS):
        return "required"

    return "mentioned"


def split_skill_segments(text):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    segments = [seg.strip() for seg in re.split(r"[\n•\-;]+", text) if seg.strip()]
    return segments


def extract_skill_records_from_text(text, source_field="unknown"):
    segments = split_skill_segments(text)
    if not segments:
        return []

    records = []

    for seg in segments:
        seg_norm = normalize_for_match(seg)

        for skill, patterns in SKILL_PATTERNS.items():
            if any(p.search(seg_norm) for p in patterns):
                records.append({
                    "skill": skill,
                    "skill_group": SKILL_TAXONOMY[skill]["group"],
                    "source_field": source_field,
                    "importance": infer_skill_importance(seg, source_field),
                    "excerpt": seg[:300],
                })

    # dedup theo skill + source + importance
    seen = set()
    out = []
    for r in records:
        key = (r["skill"], r["source_field"], r["importance"])
        if key not in seen:
            seen.add(key)
            out.append(r)

    return out


def merge_skill_records(*record_lists):
    merged = []
    seen = set()

    for records in record_lists:
        for r in records:
            key = (r["skill"], r["source_field"], r["importance"])
            if key not in seen:
                seen.add(key)
                merged.append(r)

    return merged


def list_from_records(records, key, importance_filter=None):
    vals = []

    for r in records:
        if importance_filter is not None and r.get("importance") != importance_filter:
            continue
        if r.get(key):
            vals.append(r[key])

    return deduplicate_list(vals)


def extract_job_skill_records(row):
    title_records = extract_skill_records_from_text(
        safe_str(row.get("job_title_surface")) + "\n" + safe_str(row.get("job_title_canonical")),
        source_field="title"
    )

    tag_records = extract_skill_records_from_text(
        safe_str(row.get("tags_text_canonical")),
        source_field="tags"
    )

    req_records = extract_skill_records_from_text(
        safe_str(row.get("requirements_clean_strict")),
        source_field="requirements"
    )

    desc_records = extract_skill_records_from_text(
        safe_str(row.get("description_clean_strict")),
        source_field="description"
    )

    benefit_records = extract_skill_records_from_text(
        safe_str(row.get("benefits_clean_strict")),
        source_field="benefits"
    )

    return merge_skill_records(
        title_records,
        tag_records,
        req_records,
        desc_records,
        benefit_records,
    )


def build_skill_text(xs):
    if not xs:
        return ""
    return clean_text_for_phobert(", ".join(xs))


df_clean["skill_records"] = df_clean.apply(extract_job_skill_records, axis=1)

df_clean["skills_extracted"] = df_clean["skill_records"].apply(
    lambda rs: list_from_records(rs, "skill")
)
df_clean["skill_groups_extracted"] = df_clean["skill_records"].apply(
    lambda rs: list_from_records(rs, "skill_group")
)

df_clean["skills_required"] = df_clean["skill_records"].apply(
    lambda rs: list_from_records(rs, "skill", importance_filter="required")
)
df_clean["skills_preferred"] = df_clean["skill_records"].apply(
    lambda rs: list_from_records(rs, "skill", importance_filter="preferred")
)
df_clean["skills_mentioned"] = df_clean["skill_records"].apply(
    lambda rs: list_from_records(rs, "skill", importance_filter="mentioned")
)

df_clean["skills_text_phobert"] = df_clean["skills_extracted"].apply(build_skill_text)
df_clean["skills_required_text_phobert"] = df_clean["skills_required"].apply(build_skill_text)
df_clean["skills_preferred_text_phobert"] = df_clean["skills_preferred"].apply(build_skill_text)

display(df_clean[
    [
        "job_title_raw",
        "tags_text_canonical",
        "skills_extracted",
        "skills_required",
        "skills_preferred",
        "skill_groups_extracted",
        "skills_text_phobert",
    ]
].head(10))

,job_title_raw,tags_text_canonical,skills_extracted,skills_required,skills_preferred,skill_groups_extracted,skills_text_phobert
0,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,"software tester, automation, manual, ai, it phan mem, it phan cung, thuong mai dien tu",[api],[api],[],[backend],api
1,WordPress Developer - Senior Level - (20h - 05h) Remote,"frontend developer, frontend, it phan mem",[git],[],[git],[dev_tools],git
2,Data Processing Analyst,"data analyst, data, du lieu","[python, sql, git, r, communication, problem solving, api]","[python, sql, git, r, communication, problem solving, api]",[],"[programming, database, dev_tools, soft_skills, backend]","python, sql, git, r, communication, problem solving, api"
3,Fullstack Developer ( NodeJs ) - Yêu Cầu Kinh Nghiệm Từ 4 Năm Trở Lên (CoolMate),"backend developer, backend, ai, it phan mem, thuong mai dien tu, it phan cung","[python, sql, api, aws]","[python, sql, api, aws]",[],"[programming, database, backend, cloud]","python, sql, api, aws"
4,Nhân Viên Backend Developer - Remote,"backend developer, backend, ai, it phan mem, tai chinh ngan hang","[python, sql, api, docker, communication]","[python, sql, api, docker, communication]",[],"[programming, database, backend, devops, soft_skills]","python, sql, api, docker, communication"
5,AI Engineer - Sản Phẩm AI Edtech,"ai engineer, ai, ai ml","[machine learning, deep learning, llm, docker, fastapi, flask, communication, nlp, computer vision]","[machine learning, deep learning, llm, docker, fastapi, flask, communication]",[],"[ml_concepts, ai, devops, backend, soft_skills]","machine learning, deep learning, llm, docker, fastapi, flask, communication, nlp, computer vision"
6,Chuyên Viên Phân Tích Nghiệp Vụ ( Business Analyst ),"business analyst, business analysis","[communication, sql]","[communication, sql]",[],"[soft_skills, database]","communication, sql"
7,Data Governance Specialist,"data analyst, data, du lieu","[sql, etl, data warehouse]","[sql, etl, data warehouse]",[],"[database, data_engineering, data_modeling]","sql, etl, data warehouse"
8,Middle Java Developer,"backend developer, backend","[sql, oracle, git]","[sql, oracle, git]",[],"[database, dev_tools]","sql, oracle, git"
9,Nhân Viên Kiểm Thử Phần Mềm,"software tester, automation, manual, it phan mem",[communication],[communication],[],[soft_skills],communication


In [109]:
# =========================
# SKILL TAXONOMY - QA / STATS
# =========================

job_skill_rows = []
for _, row in df_clean.iterrows():
    for r in row["skill_records"]:
        job_skill_rows.append({
            "job_url": row.get("job_url"),
            "job_title_display": row.get("job_title_display"),
            "job_title_canonical": row.get("job_title_canonical"),
            "job_family": row.get("job_family"),
            "location_city": row.get("location_city"),
            "skill": r["skill"],
            "skill_group": r["skill_group"],
            "source_field": r["source_field"],
            "importance": r["importance"],
            "excerpt": r["excerpt"],
        })

job_skill_map_df = pd.DataFrame(job_skill_rows)
display(job_skill_map_df.head(10))

if len(job_skill_map_df) > 0:
    role_skill_stats_df = (
        job_skill_map_df.groupby(["job_family", "skill", "importance"])
        .size()
        .reset_index(name="job_count")
        .sort_values(["job_family", "job_count"], ascending=[True, False])
    )
else:
    role_skill_stats_df = pd.DataFrame(columns=["job_family", "skill", "importance", "job_count"])

display(role_skill_stats_df.head(20))

print("Tỷ lệ job không extract được skill:", (df_clean["skills_extracted"].apply(len) == 0).mean())

,job_url,job_title_display,job_title_canonical,job_family,location_city,skill,skill_group,source_field,importance,excerpt
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,TP. Hà Nội,api,backend,requirements,required,tốt nghiệp chuyên ngành cntt khoa học máy tính hoặc lĩnh vực liên quan. có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. có kinh nghiệm về sản phẩm product android nắm...
1,https://www.topcv.vn/brand/cennext/tuyen-dung/wordpress-developer-senior-level-20h-05h-remote-j1867723.html,WordPress Developer - Senior Level,wordpress developer - level,software_engineering,"TP. Hà Nội, TP. Đà Nẵng, Đồng Nai, TP. Hồ Chí Minh",git,dev_tools,requirements,preferred,page seo hygiene. english b1 or higher read docs write short updates detail oriented and reliable. nice to have build tools npm babel webpack grunt gulp performance/security basics wp rocket wordf...
2,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,python,programming,requirements,required,haves strong understanding of data analytics data processing and scripting. experience working with python r mysql and other sql languages . proficiency with git a plus communication interpersonal...
3,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,sql,database,requirements,required,haves strong understanding of data analytics data processing and scripting. experience working with python r mysql and other sql languages . proficiency with git a plus communication interpersonal...
4,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,git,dev_tools,requirements,required,haves strong understanding of data analytics data processing and scripting. experience working with python r mysql and other sql languages . proficiency with git a plus communication interpersonal...
5,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,r,programming,requirements,required,haves strong understanding of data analytics data processing and scripting. experience working with python r mysql and other sql languages . proficiency with git a plus communication interpersonal...
6,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,communication,soft_skills,requirements,required,haves strong understanding of data analytics data processing and scripting. experience working with python r mysql and other sql languages . proficiency with git a plus communication interpersonal...
7,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,python,programming,description,required,get to know the role we are looking for a new analyst for one of our data processing teams. we believe a successful candidate has problem solving and python programming skills but if you believe y...
8,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,Data Processing Analyst,data labeling specialist,data_governance_quality,TP. Hồ Chí Minh,problem solving,soft_skills,description,required,get to know the role we are looking for a new analyst for one of our data processing teams. we believe a successful candidate has problem solving and python programming skills but if you believe y...

,job_family,skill,importance,job_count
33,cloud_devops_sre,kubernetes,required,62
18,cloud_devops_sre,docker,required,55
6,cloud_devops_sre,aws,required,52
26,cloud_devops_sre,git,required,49
31,cloud_devops_sre,kubernetes,mentioned,48
4,cloud_devops_sre,aws,mentioned,45
44,cloud_devops_sre,python,required,41
12,cloud_devops_sre,communication,required,35
9,cloud_devops_sre,azure,required,28
16,cloud_devops_sre,docker,mentioned,28


Tỷ lệ job không extract được skill: 0.08463100880162491


In [110]:
# =========================
# FINAL JOB TEXT BUILDERS
# =========================

def format_salary_brief(row):
    mn = row.get("salary_min_vnd_month")
    mx = row.get("salary_max_vnd_month")

    if pd.notna(mn) and pd.notna(mx):
        return f"{int(mn):,}-{int(mx):,} VND/tháng"
    if pd.notna(mn):
        return f"từ {int(mn):,} VND/tháng"
    if row.get("salary_is_negotiable") is True:
        return "thỏa thuận"
    return ""


def format_experience_brief(row):
    mn = row.get("experience_min_years")
    mx = row.get("experience_max_years")
    exp_type = safe_str(row.get("experience_type"))

    if exp_type == "no_experience":
        return "không yêu cầu kinh nghiệm"

    if pd.notna(mn) and pd.notna(mx):
        if mn == mx:
            return f"{mn:g} năm"
        return f"{mn:g}-{mx:g} năm"

    if pd.notna(mn):
        return f"từ {mn:g} năm"

    if pd.notna(mx):
        return f"dưới {mx:g} năm"

    return ""


def build_job_text_sparse(row):
    parts = []

    for field in [
        row.get("job_title_canonical"),
        row.get("job_family"),
        row.get("job_level_norm"),
        row.get("employment_type_norm"),
        row.get("education_level_norm"),
        row.get("tags_text_canonical"),
        row.get("skills_text_phobert"),
        row.get("skills_required_text_phobert"),
        row.get("requirements_clean_strict"),
        row.get("description_clean_strict"),
    ]:
        s = safe_str(field)
        if s and s != "unknown":
            parts.append(s)

    return "\n".join(parts).strip()


def build_job_text_phobert_match(row):
    parts = []

    title = row.get("job_title_for_phobert")
    family = row.get("job_family")
    job_level = row.get("job_level_norm")
    employment_type = row.get("employment_type_norm")
    education_level = row.get("education_level_norm")
    required_skills = row.get("skills_required_text_phobert")
    preferred_skills = row.get("skills_preferred_text_phobert")
    tags_text = row.get("tags_text_for_model")
    exp_brief = format_experience_brief(row)
    location = row.get("location_city")
    work_mode = row.get("work_mode")
    req = truncate_by_words(row.get("requirements_clean_phobert"), 160)

    if title:
        parts.append(f"Vị trí: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm nghề: {family}")
    if job_level and job_level != "unknown":
        parts.append(f"Cấp độ công việc: {job_level}")
    if employment_type and employment_type != "unknown":
        parts.append(f"Hình thức tuyển dụng: {employment_type}")
    if education_level and education_level != "unknown":
        parts.append(f"Học vấn: {education_level}")
    if required_skills:
        parts.append(f"Kỹ năng bắt buộc: {required_skills}")
    if preferred_skills:
        parts.append(f"Kỹ năng ưu tiên: {preferred_skills}")
    if tags_text:
        parts.append(f"Tag công việc: {tags_text}")
    if exp_brief:
        parts.append(f"Kinh nghiệm: {exp_brief}")
    if location:
        parts.append(f"Địa điểm: {location}")
    if work_mode and work_mode != "unknown":
        parts.append(f"Hình thức làm việc: {work_mode}")
    if req:
        parts.append(f"Yêu cầu chính: {req}")

    return "\n".join(parts).strip()


def build_job_text_phobert_chatbot(row):
    parts = []

    title = row.get("job_title_for_phobert")
    family = row.get("job_family")
    job_level = row.get("job_level_norm")
    employment_type = row.get("employment_type_norm")
    education_level = row.get("education_level_norm")
    location = row.get("location_city")
    work_mode = row.get("work_mode")
    salary_brief = format_salary_brief(row)
    exp_brief = format_experience_brief(row)
    tags_text = row.get("tags_text_for_model")
    working_address = row.get("working_address_clean")
    required_skills = row.get("skills_required_text_phobert")
    preferred_skills = row.get("skills_preferred_text_phobert")
    requirements_text = truncate_by_words(row.get("requirements_clean_phobert"), 220)
    description_text = truncate_by_words(row.get("description_clean_phobert"), 220)
    benefits_text = truncate_by_words(row.get("benefits_clean_phobert"), 120)

    if title:
        parts.append(f"Vị trí tuyển dụng: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm công việc: {family}")
    if job_level and job_level != "unknown":
        parts.append(f"Cấp độ công việc: {job_level}")
    if employment_type and employment_type != "unknown":
        parts.append(f"Loại hình tuyển dụng: {employment_type}")
    if education_level and education_level != "unknown":
        parts.append(f"Yêu cầu học vấn: {education_level}")
    if location:
        parts.append(f"Địa điểm: {location}")
    if working_address:
        parts.append(f"Nơi làm việc: {working_address}")
    if work_mode and work_mode != "unknown":
        parts.append(f"Hình thức làm việc: {work_mode}")
    if salary_brief:
        parts.append(f"Mức lương: {salary_brief}")
    if exp_brief:
        parts.append(f"Kinh nghiệm: {exp_brief}")
    if tags_text:
        parts.append(f"Tags liên quan: {tags_text}")
    if required_skills:
        parts.append(f"Kỹ năng bắt buộc: {required_skills}")
    if preferred_skills:
        parts.append(f"Kỹ năng ưu tiên: {preferred_skills}")
    if requirements_text:
        parts.append(f"Yêu cầu:\n{requirements_text}")
    if description_text:
        parts.append(f"Mô tả công việc:\n{description_text}")
    if benefits_text:
        parts.append(f"Quyền lợi:\n{benefits_text}")

    return "\n\n".join(parts).strip()

In [111]:
df_clean["job_text_sparse"] = df_clean.apply(build_job_text_sparse, axis=1)
df_clean["job_text_phobert_match"] = df_clean.apply(build_job_text_phobert_match, axis=1)
df_clean["job_text_phobert_chatbot"] = df_clean.apply(build_job_text_phobert_chatbot, axis=1)

df_clean["dense_encoder_route"] = "phobert"
df_clean["dense_similarity_metric"] = "cosine"

display(df_clean[
    [
        "job_title_raw",
        "job_text_sparse",
        "job_text_phobert_match",
        "job_text_phobert_chatbot",
    ]
].head(3))

,job_title_raw,job_text_sparse,job_text_phobert_match,job_text_phobert_chatbot
0,Nhân Viên Tester Mobile App ( Android )- Đi Làm Ngay,"automation tester\nqa_testing\njunior\nfull_time\nbachelor\nsoftware tester, automation, manual, ai, it phan mem, it phan cung, thuong mai dien tu\napi\napi\ntốt nghiệp chuyên ngành cntt khoa học ...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: api\nTag công việc: Ch...,Vị trí tuyển dụng: Nhân Viên Tester Mobile App automation tester\n\nNhóm công việc: qa_testing\n\nCấp độ công việc: junior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu học vấn: bachelor\n\nĐịa đi...
1,WordPress Developer - Senior Level - (20h - 05h) Remote,"wordpress developer - level\nsoftware_engineering\nsenior\nfull_time\ncollege\nfrontend developer, frontend, it phan mem\ngit\n3 years of hands-on wordpress publishing/site build experience. good ...",Vị trí: WordPress Developer - Senior Level wordpress developer - level\nNhóm nghề: software_engineering\nCấp độ công việc: senior\nHình thức tuyển dụng: full_time\nHọc vấn: college\nKỹ năng ưu tiê...,Vị trí tuyển dụng: WordPress Developer - Senior Level wordpress developer - level\n\nNhóm công việc: software_engineering\n\nCấp độ công việc: senior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu ...
2,Data Processing Analyst,"data labeling specialist\ndata_governance_quality\njunior\nfull_time\nbachelor\ndata analyst, data, du lieu\npython, sql, git, r, communication, problem solving, api\npython, sql, git, r, communic...",Vị trí: Data Processing Analyst data labeling specialist\nNhóm nghề: data_governance_quality\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: python...,Vị trí tuyển dụng: Data Processing Analyst data labeling specialist\n\nNhóm công việc: data_governance_quality\n\nCấp độ công việc: junior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu học vấn: ba...


In [112]:
# =========================
# SECTION CHUNKING + SECTION RECORDS
# =========================

def split_long_text(text, max_chars=700, overlap=80):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""

    for para in paras:
        if len(current) + len(para) + 2 <= max_chars:
            current = f"{current}\n\n{para}".strip()
        else:
            if current:
                chunks.append(current)

            if len(para) <= max_chars:
                current = para
            else:
                start = 0
                while start < len(para):
                    end = min(start + max_chars, len(para))
                    chunk = para[start:end]
                    chunks.append(chunk.strip())

                    # overlap thật sự
                    if end >= len(para):
                        break
                    start = max(0, end - overlap)

                current = ""

    if current:
        chunks.append(current)

    return [c.strip() for c in chunks if c.strip()]


SECTION_PRIORITY = {
    "title": 5.0,
    "requirements": 4.5,
    "description": 4.0,
    "benefits": 3.0,
    "company": 2.5,
}


def build_chunk_text_phobert(row, section_type: str, chunk_text: str):
    title = safe_str(row.get("job_title_for_phobert"))
    family = safe_str(row.get("job_family"))
    job_level = safe_str(row.get("job_level_norm"))
    location = safe_str(row.get("location_city"))
    work_mode = safe_str(row.get("work_mode"))
    required_skills = safe_str(row.get("skills_required_text_phobert"))

    section_map = {
        "title": "Tiêu đề",
        "requirements": "Yêu cầu",
        "description": "Mô tả công việc",
        "benefits": "Quyền lợi",
        "company": "Giới thiệu công ty",
    }
    section_label = section_map.get(section_type, section_type)

    parts = []

    if title:
        parts.append(f"Vị trí: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm nghề: {family}")
    if job_level and job_level != "unknown":
        parts.append(f"Cấp độ: {job_level}")
    if location:
        parts.append(f"Địa điểm: {location}")
    if work_mode and work_mode != "unknown":
        parts.append(f"Hình thức làm việc: {work_mode}")
    if required_skills and section_type in {"title", "requirements", "description"}:
        parts.append(f"Kỹ năng chính: {required_skills}")

    parts.append(f"Phần: {section_label}")
    parts.append(truncate_by_words(chunk_text, 180))

    return "\n".join([p for p in parts if p]).strip()


def build_job_section_records(row):
    rows = []

    section_map = {
        "title": safe_str(row.get("job_title_for_phobert")),
        "requirements": safe_str(row.get("requirements_clean_phobert")),
        "description": safe_str(row.get("description_clean_phobert")),
        "benefits": safe_str(row.get("benefits_clean_phobert")),
        "company": safe_str(row.get("company_description_clean_phobert")),
    }

    for section_type, text in section_map.items():
        if not text:
            continue

        chunks = [text] if section_type == "title" else split_long_text(text, max_chars=700, overlap=80)

        for chunk_order, chunk_text in enumerate(chunks):
            rows.append({
                "job_url": row.get("job_url"),
                "job_title_display": row.get("job_title_display"),
                "job_title_canonical": row.get("job_title_canonical"),
                "job_family": row.get("job_family"),
                "job_level_norm": row.get("job_level_norm"),
                "location_city": row.get("location_city"),
                "work_mode": row.get("work_mode"),
                "section_type": section_type,
                "chunk_order": chunk_order,
                "section_priority": SECTION_PRIORITY.get(section_type, 1.0),
                "chunk_text_raw": chunk_text,
                "chunk_text_phobert": build_chunk_text_phobert(row, section_type, chunk_text),
            })

    return rows


section_rows = []
for _, row in df_clean.iterrows():
    section_rows.extend(build_job_section_records(row))

job_sections_df = pd.DataFrame(section_rows)

display(job_sections_df.head(5))
print("job_sections_df shape:", job_sections_df.shape)

,job_url,job_title_display,job_title_canonical,job_family,job_level_norm,location_city,work_mode,section_type,chunk_order,section_priority,chunk_text_raw,chunk_text_phobert
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,title,0,5.0,Nhân Viên Tester Mobile App automation tester,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Tiêu đề\nNhân Viên Tester ...
1,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,requirements,0,4.5,"Tốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan. Có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. Có kinh nghiệm về sản phẩm Product android Nắ...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nTốt nghiệp chuyên...
2,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,requirements,1,4.5,"est mobile game hoặc app có lượng người dùng lớn\nTốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan.\nCó kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói r...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nest mobile game h...
3,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,requirements,2,4.5,"PI căn bản (GET/POST, hiểu JSON)\nCó kinh nghiệm test mobile game hoặc app có lượng người dùng lớn",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nPI căn bản (GET/P...
4,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,description,0,4.0,"Phân tích yêu cầu sản phẩm và thiết kế để xây dựng test case, test plan cho ứng dụng mobile. Thực hiện kiểm thử ứng dụng trên nhiều thiết bị thực tế và trình giả lập (emulator/simulator). Kiểm thử...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Mô tả công việc\nPhân tích...


job_sections_df shape: (11677, 12)


In [113]:
# =========================
# PHOBERT PREP + EMBEDDING UTILS
# =========================

def maybe_segment_vi_text(text):
    text = safe_str(text).strip()
    if not text:
        return ""

    # tránh segment các chuỗi quá ngắn / quá ít từ
    if len(text.split()) <= 2:
        return text

    if HAS_UNDERTHESEA:
        try:
            segmented = word_tokenize(text, format="text")
            segmented = safe_str(segmented).strip()
            return segmented if segmented else text
        except Exception:
            return text

    return text


def prepare_phobert_text(text: str) -> str:
    text = safe_str(text)
    if not text.strip():
        return "[EMPTY]"

    text = clean_text_for_phobert(text)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return "[EMPTY]"

    text = maybe_segment_vi_text(text)
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else "[EMPTY]"


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    masked_sum = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    mask_sum = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

    return masked_sum / mask_sum


def l2_normalize_embeddings(x):
    if isinstance(x, torch.Tensor):
        return torch.nn.functional.normalize(x, p=2, dim=1)
    x = np.asarray(x)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return x / norms


def cosine_similarity_matrix(query_emb, doc_embs):
    query_emb = np.asarray(query_emb)
    doc_embs = np.asarray(doc_embs)

    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)
    if doc_embs.ndim == 1:
        doc_embs = doc_embs.reshape(1, -1)

    query_emb = l2_normalize_embeddings(query_emb)
    doc_embs = l2_normalize_embeddings(doc_embs)

    return np.dot(query_emb, doc_embs.T)

In [114]:
df_clean["job_text_phobert_match_prepared"] = df_clean["job_text_phobert_match"].apply(prepare_phobert_text)
df_clean["job_text_phobert_chatbot_prepared"] = df_clean["job_text_phobert_chatbot"].apply(prepare_phobert_text)

job_sections_df["chunk_text_phobert_prepared"] = job_sections_df["chunk_text_phobert"].apply(prepare_phobert_text)

In [115]:
display(df_clean[[
    "job_text_phobert_match",
    "job_text_phobert_match_prepared"
]].head(5))

display(job_sections_df[[
    "chunk_text_phobert",
    "chunk_text_phobert_prepared"
]].head(5))

,job_text_phobert_match,job_text_phobert_match_prepared
0,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: api\nTag công việc: Ch...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ công_việc : junior Hình_thức tuyển_dụng : full_time Học_vấn : bachelor Kỹ_năng bắt_buộc : api_Tag công_việc : C...
1,Vị trí: WordPress Developer - Senior Level wordpress developer - level\nNhóm nghề: software_engineering\nCấp độ công việc: senior\nHình thức tuyển dụng: full_time\nHọc vấn: college\nKỹ năng ưu tiê...,Vị_trí : WordPress_Developer - Senior Level wordpress developer - level Nhóm nghề : software_engineering Cấp_độ công_việc : senior Hình_thức tuyển_dụng : full_time Học_vấn : college Kỹ_năng ưu_tiê...
2,Vị trí: Data Processing Analyst data labeling specialist\nNhóm nghề: data_governance_quality\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: python...,Vị_trí : Data_Processing Analyst data labeling specialist Nhóm nghề : data_governance_quality Cấp_độ công_việc : junior Hình_thức tuyển_dụng : full_time Học_vấn : bachelor Kỹ_năng bắt_buộc : pytho...
3,"Vị trí: Fullstack Developer (NodeJs) fullstack developer\nNhóm nghề: software_engineering\nCấp độ công việc: senior\nHình thức tuyển dụng: full_time\nHọc vấn: college\nKỹ năng bắt buộc: python, sq...",Vị_trí : Fullstack_Developer ( NodeJs ) fullstack_developer Nhóm nghề : software_engineering Cấp_độ công_việc : senior Hình_thức tuyển_dụng : full_time Học_vấn : college Kỹ_năng bắt_buộc : python ...
4,"Vị trí: Nhân Viên Backend Developer backend developer\nNhóm nghề: software_engineering\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: college\nKỹ năng bắt buộc: python, sql, ...","Vị_trí : Nhân_Viên Backend Developer backend_developer Nhóm nghề : software_engineering Cấp_độ công_việc : junior Hình_thức tuyển_dụng : full_time Học_vấn : college Kỹ_năng bắt_buộc : python , sql..."


,chunk_text_phobert,chunk_text_phobert_prepared
0,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Tiêu đề\nNhân Viên Tester ...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Tiêu_đề Nhân_Viên Tester_...
1,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nTốt nghiệp chuyên...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Yêu_cầu Tốt_nghiệp chuyên...
2,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nest mobile game h...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Yêu_cầu est mobile game h...
3,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nPI căn bản (GET/P...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Yêu_cầu PI căn_bản ( GET ...
4,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Mô tả công việc\nPhân tích...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Mô_tả công_việc Phân_tích...


In [116]:
# =========================
# PREPARED TEXT COLUMNS
# =========================

df_clean["job_text_phobert_match_prepared"] = df_clean["job_text_phobert_match"].apply(prepare_phobert_text)
df_clean["job_text_phobert_chatbot_prepared"] = df_clean["job_text_phobert_chatbot"].apply(prepare_phobert_text)

if len(job_sections_df) > 0:
    job_sections_df["chunk_text_phobert_prepared"] = job_sections_df["chunk_text_phobert"].apply(prepare_phobert_text)
else:
    job_sections_df["chunk_text_phobert_prepared"] = ""

display(df_clean[
    ["job_text_phobert_match", "job_text_phobert_match_prepared"]
].head(3))

display(job_sections_df[
    ["chunk_text_phobert", "chunk_text_phobert_prepared"]
].head(3))

,job_text_phobert_match,job_text_phobert_match_prepared
0,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: api\nTag công việc: Ch...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ công_việc : junior Hình_thức tuyển_dụng : full_time Học_vấn : bachelor Kỹ_năng bắt_buộc : api_Tag công_việc : C...
1,Vị trí: WordPress Developer - Senior Level wordpress developer - level\nNhóm nghề: software_engineering\nCấp độ công việc: senior\nHình thức tuyển dụng: full_time\nHọc vấn: college\nKỹ năng ưu tiê...,Vị_trí : WordPress_Developer - Senior Level wordpress developer - level Nhóm nghề : software_engineering Cấp_độ công_việc : senior Hình_thức tuyển_dụng : full_time Học_vấn : college Kỹ_năng ưu_tiê...
2,Vị trí: Data Processing Analyst data labeling specialist\nNhóm nghề: data_governance_quality\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: python...,Vị_trí : Data_Processing Analyst data labeling specialist Nhóm nghề : data_governance_quality Cấp_độ công_việc : junior Hình_thức tuyển_dụng : full_time Học_vấn : bachelor Kỹ_năng bắt_buộc : pytho...


,chunk_text_phobert,chunk_text_phobert_prepared
0,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Tiêu đề\nNhân Viên Tester ...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Tiêu_đề Nhân_Viên Tester_...
1,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nTốt nghiệp chuyên...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Yêu_cầu Tốt_nghiệp chuyên...
2,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nest mobile game h...,Vị_trí : Nhân_Viên Tester_Mobile App automation_tester Nhóm nghề : qa_testing Cấp_độ : junior Địa_điểm : TP. Hà_Nội Hình_thức làm_việc : onsite Kỹ_năng chính : api Phần : Yêu_cầu est mobile game h...


In [117]:
# =========================
# LOAD PHOBERT + ENCODER
# =========================

if RUN_EMBEDDING:
    tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL_NAME)
    model = AutoModel.from_pretrained(PHOBERT_MODEL_NAME)
    model.to(DEVICE)
    model.eval()

    def encode_phobert_texts(
        texts,
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_MATCH,
        already_prepared=False
    ):
        texts = list(texts) if texts is not None else []
        if len(texts) == 0:
            return np.empty((0, model.config.hidden_size), dtype=np.float32)

        if already_prepared:
            prepared = [safe_str(t).strip() if safe_str(t).strip() else "[EMPTY]" for t in texts]
        else:
            prepared = [prepare_phobert_text(t) for t in texts]

        embeddings = []

        for i in tqdm(range(0, len(prepared), batch_size), desc="Encoding"):
            batch = prepared[i:i + batch_size]

            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(DEVICE)
            attention_mask = encoded["attention_mask"].to(DEVICE)

            with torch.no_grad():
                output = model(input_ids=input_ids, attention_mask=attention_mask)

            batch_emb = mean_pooling(output, attention_mask)

            if NORMALIZE_EMBEDDINGS:
                batch_emb = F.normalize(batch_emb, p=2, dim=1)

            embeddings.append(batch_emb.cpu().numpy())

        return np.vstack(embeddings)

    print("[INFO] PhoBERT loaded successfully.")
else:
    print("[INFO] RUN_EMBEDDING=False -> skip model loading.")

[INFO] RUN_EMBEDDING=False -> skip model loading.


In [118]:
# =========================
# SECTION-LEVEL + JOB-LEVEL EMBEDDING
# =========================

section_dense_embeddings = None
job_dense_embeddings = None

# ----- Section-level -----
if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and len(job_sections_df) > 0:
    job_sections_df["section_embedding_row_id"] = np.arange(len(job_sections_df))

    section_dense_embeddings = encode_phobert_texts(
        job_sections_df["chunk_text_phobert_prepared"].fillna("").tolist(),
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_CHUNK,
        already_prepared=True,
    )

    job_sections_df["has_dense_embedding"] = 1
    print("section_dense_embeddings shape:", section_dense_embeddings.shape)
else:
    job_sections_df["has_dense_embedding"] = 0
    print("[INFO] Skip section-level embedding.")

# ----- Job-level -----
df_clean["job_embedding_row_id"] = np.arange(len(df_clean))

if RUN_EMBEDDING and len(df_clean) > 0:
    job_dense_embeddings = encode_phobert_texts(
        df_clean["job_text_phobert_match_prepared"].fillna("").tolist(),
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_MATCH,
        already_prepared=True,
    )

    df_clean["has_dense_embedding"] = 1
    print("job_dense_embeddings shape:", job_dense_embeddings.shape)
else:
    df_clean["has_dense_embedding"] = 0
    print("[INFO] Skip job-level embedding.")

[INFO] Skip section-level embedding.
[INFO] Skip job-level embedding.


In [119]:
print("RUN_EMBEDDING =", RUN_EMBEDDING)
print("encode_phobert_texts exists =", "encode_phobert_texts" in globals())
print("job_dense_embeddings is None =", job_dense_embeddings is None if "job_dense_embeddings" in globals() else "NOT DEFINED")

RUN_EMBEDDING = False
encode_phobert_texts exists = False
job_dense_embeddings is None = True


In [130]:
# =========================
# DENSE RETRIEVAL
# =========================

def encode_query_for_matching(query: str, max_length=PHOBERT_MAX_LENGTH_MATCH):
    q = encode_phobert_texts([query], batch_size=1, max_length=max_length, already_prepared=False)
    return q[0]


def retrieve_top_jobs(query: str, top_k: int = 10):
    if job_dense_embeddings is None:
        raise RuntimeError("job_dense_embeddings is None. Hãy bật RUN_SECTION_EMBEDDING=True.")

    q_emb = encode_query_for_matching(query, max_length=PHOBERT_MAX_LENGTH_MATCH)
    scores = job_dense_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    cols = [
        "job_url",
        "job_title_display",
        "job_family",
        "job_level_norm",
        "location_city",
        "work_mode",
        "skills_required",
        "skills_preferred",
        "job_text_phobert_match",
    ]
    out = df_clean.iloc[top_idx][cols].copy()
    out["cosine_score"] = scores[top_idx]
    return out.reset_index(drop=True)


def retrieve_top_sections(query: str, top_k: int = 10):
    if section_dense_embeddings is None:
        raise RuntimeError("section_dense_embeddings is None. Hãy bật RUN_SECTION_EMBEDDING=True.")

    q_emb = encode_query_for_matching(query, max_length=PHOBERT_MAX_LENGTH_CHUNK)
    scores = section_dense_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    cols = [
        "job_url",
        "job_title_display",
        "job_family",
        "job_level_norm",
        "location_city",
        "section_type",
        "chunk_order",
        "section_priority",
        "chunk_text_raw",
    ]
    out = job_sections_df.iloc[top_idx][cols].copy()
    out["cosine_score"] = scores[top_idx]
    return out.reset_index(drop=True)


demo_query = "Data Analyst cần SQL Power BI và kinh nghiệm phân tích dữ liệu"

try:
    display(retrieve_top_jobs(demo_query, top_k=5))
    display(retrieve_top_sections(demo_query, top_k=5))
except Exception as e:
    print("[INFO] Demo retrieval skipped:", e)

[INFO] Demo retrieval skipped: job_dense_embeddings is None. Hãy bật RUN_SECTION_EMBEDDING=True.


In [131]:
DOWNSTREAM_FIELD_GUIDE = {
    "tfidf_input": "job_text_sparse",
    "phobert_matching_input": "job_text_phobert_match",
    "phobert_chatbot_input": "job_text_phobert_chatbot",
    "chatbot_chunk_table": "job_sections_df",
    "chatbot_chunk_text_field": "chunk_text_phobert",
    "skill_table": "job_skill_map_df",
    "role_skill_stats": "role_skill_stats_df",
    "retrieval_metric": "cosine_similarity_on_l2_normalized_embeddings",
    "job_embedding_id_field": "job_embedding_row_id",
    "section_embedding_id_field": "section_embedding_row_id",
    "location_field": "location_city",
    "working_address_field": "working_address_clean",
}

pd.Series(DOWNSTREAM_FIELD_GUIDE)

tfidf_input                                                 job_text_sparse
phobert_matching_input                               job_text_phobert_match
phobert_chatbot_input                              job_text_phobert_chatbot
chatbot_chunk_table                                         job_sections_df
chatbot_chunk_text_field                                 chunk_text_phobert
skill_table                                                job_skill_map_df
role_skill_stats                                        role_skill_stats_df
retrieval_metric              cosine_similarity_on_l2_normalized_embeddings
job_embedding_id_field                                 job_embedding_row_id
section_embedding_id_field                         section_embedding_row_id
location_field                                                location_city
working_address_field                                 working_address_clean
dtype: str

In [132]:
matching_cols = [
    "job_url", "job_id",
    "job_title_display", "job_title_canonical", "job_family",
    "job_level_norm",
    "location_city", "working_address_clean", "is_multi_location", "work_mode",
    "salary_min_vnd_month", "salary_max_vnd_month", "salary_is_negotiable", "salary_bucket",
    "experience_min_years", "experience_max_years", "experience_type",
    "education_level_norm", "employment_type_norm",
    "skills_extracted", "skills_required", "skills_preferred", "skill_groups_extracted",
    "tags_role_list", "tags_domain_list", "tags_specialty_list",
    "job_text_sparse", "job_text_phobert_match", "job_text_phobert_chatbot",
    "job_embedding_row_id", "has_dense_embedding", "dense_encoder_route", "dense_similarity_metric",
]

matching_cols = [c for c in matching_cols if c in df_clean.columns]
df_matching_ready = df_clean[matching_cols].copy()
df_matching_ready["dense_model_name"] = PHOBERT_MODEL_NAME
df_matching_ready["dense_similarity_metric"] = "cosine"

display(df_matching_ready.head(3))
print(df_matching_ready.shape)

,job_url,job_id,job_title_display,job_title_canonical,job_family,job_level_norm,location_city,working_address_clean,is_multi_location,work_mode,salary_min_vnd_month,salary_max_vnd_month,salary_is_negotiable,salary_bucket,experience_min_years,experience_max_years,experience_type,education_level_norm,employment_type_norm,skills_extracted,skills_required,skills_preferred,skill_groups_extracted,tags_role_list,tags_domain_list,tags_specialty_list,job_text_sparse,job_text_phobert_match,job_text_phobert_chatbot,job_embedding_row_id,has_dense_embedding,dense_encoder_route,dense_similarity_metric,dense_model_name
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,None,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,"Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân",False,onsite,NaN,NaN,True,negotiable,1.0,1.0,fixed,bachelor,full_time,[api],[api],[],[backend],[software tester],"[it phan mem, it phan cung, thuong mai dien tu]","[automation, manual, ai]","automation tester\nqa_testing\njunior\nfull_time\nbachelor\nsoftware tester, automation, manual, ai, it phan mem, it phan cung, thuong mai dien tu\napi\napi\ntốt nghiệp chuyên ngành cntt khoa học ...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: api\nTag công việc: Ch...,Vị trí tuyển dụng: Nhân Viên Tester Mobile App automation tester\n\nNhóm công việc: qa_testing\n\nCấp độ công việc: junior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu học vấn: bachelor\n\nĐịa đi...,0,0,phobert,cosine,vinai/phobert-base
1,https://www.topcv.vn/brand/cennext/tuyen-dung/wordpress-developer-senior-level-20h-05h-remote-j1867723.html,None,WordPress Developer - Senior Level,wordpress developer - level,software_engineering,senior,"TP. Hà Nội, TP. Đà Nẵng, Đồng Nai, TP. Hồ Chí Minh",Đà Nẵng: - Đồng Nai,True,remote,12000000.0,18000000.0,False,10m_20m,3.0,3.0,fixed,college,full_time,[git],[],[git],[dev_tools],[frontend developer],[it phan mem],[frontend],"wordpress developer - level\nsoftware_engineering\nsenior\nfull_time\ncollege\nfrontend developer, frontend, it phan mem\ngit\n3 years of hands-on wordpress publishing/site build experience. good ...",Vị trí: WordPress Developer - Senior Level wordpress developer - level\nNhóm nghề: software_engineering\nCấp độ công việc: senior\nHình thức tuyển dụng: full_time\nHọc vấn: college\nKỹ năng ưu tiê...,Vị trí tuyển dụng: WordPress Developer - Senior Level wordpress developer - level\n\nNhóm công việc: software_engineering\n\nCấp độ công việc: senior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu ...,1,0,phobert,cosine,vinai/phobert-base
2,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,None,Data Processing Analyst,data labeling specialist,data_governance_quality,junior,TP. Hồ Chí Minh,"Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận Hồ Chí Minh",False,unknown,25000000.0,30000000.0,False,30m_50m,2.0,2.0,fixed,bachelor,full_time,"[python, sql, git, r, communication, problem solving, api]","[python, sql, git, r, communication, problem solving, api]",[],"[programming, database, dev_tools, soft_skills, backend]",[data analyst],[du lieu],[data],"data labeling specialist\ndata_governance_quality\njunior\nfull_time\nbachelor\ndata analyst, data, du lieu\npython, sql, git, r, communication, problem solving, api\npython, sql, git, r, communic...",Vị trí: Data Processing Analyst data labeling specialist\nNhóm nghề: data_governance_quality\nCấp độ công việc: junior\nHình thức tuyển dụng: full_time\nHọc vấn: bachelor\nKỹ năng bắt buộc: python...,Vị trí tuyển dụng: Data Processing Analyst data labeling specialist\n\nNhóm công việc: data_governance_quality\n\nCấp độ công việc: junior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu học vấn: ba...,2,0,phobert,cosine,vinai/phobert-base


(1477, 34)


In [133]:
chatbot_cols = [
    "job_url", "job_id",
    "job_title_display", "job_title_canonical", "job_family", "job_level_norm",
    "location_city", "working_address_clean", "is_multi_location", "work_mode",
    "salary_min_vnd_month", "salary_max_vnd_month", "salary_is_negotiable", "salary_bucket",
    "experience_min_years", "experience_max_years", "experience_type",
    "education_level_norm", "employment_type_norm",
    "skills_extracted", "skills_required", "skills_preferred",
    "tags_role_list", "tags_domain_list", "tags_specialty_list",
    "requirements_clean_phobert", "description_clean_phobert", "benefits_clean_phobert",
    "job_text_phobert_chatbot",
    "job_embedding_row_id", "has_dense_embedding"
]

chatbot_cols = [c for c in chatbot_cols if c in df_clean.columns]
df_chatbot_ready = df_clean[chatbot_cols].copy()

display(df_chatbot_ready.head(3))
print(df_chatbot_ready.shape)

,job_url,job_id,job_title_display,job_title_canonical,job_family,job_level_norm,location_city,working_address_clean,is_multi_location,work_mode,salary_min_vnd_month,salary_max_vnd_month,salary_is_negotiable,salary_bucket,experience_min_years,experience_max_years,experience_type,education_level_norm,employment_type_norm,skills_extracted,skills_required,skills_preferred,tags_role_list,tags_domain_list,tags_specialty_list,requirements_clean_phobert,description_clean_phobert,benefits_clean_phobert,job_text_phobert_chatbot,job_embedding_row_id,has_dense_embedding
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,None,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,"Hà Nội: 116 Nhân Hòa, Thanh Xuân, Phường Thanh Xuân",False,onsite,NaN,NaN,True,negotiable,1.0,1.0,fixed,bachelor,full_time,[api],[api],[],[software tester],"[it phan mem, it phan cung, thuong mai dien tu]","[automation, manual, ai]","Tốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan. Có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. Có kinh nghiệm về sản phẩm Product android Nắ...","Phân tích yêu cầu sản phẩm và thiết kế để xây dựng test case, test plan cho ứng dụng mobile. Thực hiện kiểm thử ứng dụng trên nhiều thiết bị thực tế và trình giả lập (emulator/simulator). Kiểm thử...","Được sử dụng thiết bị test (smartphone thật, tool nội bộ). BHXH - BHYT - BHTN đầy đủ theo luật. Du lịch, teambuilding, sinh nhật, thưởng lễ tết, đào tạo kỹ năng. Lộ trình phát triển rõ ràng: Cơ hộ...",Vị trí tuyển dụng: Nhân Viên Tester Mobile App automation tester\n\nNhóm công việc: qa_testing\n\nCấp độ công việc: junior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu học vấn: bachelor\n\nĐịa đi...,0,0
1,https://www.topcv.vn/brand/cennext/tuyen-dung/wordpress-developer-senior-level-20h-05h-remote-j1867723.html,None,WordPress Developer - Senior Level,wordpress developer - level,software_engineering,senior,"TP. Hà Nội, TP. Đà Nẵng, Đồng Nai, TP. Hồ Chí Minh",Đà Nẵng: - Đồng Nai,True,remote,12000000.0,18000000.0,False,10m_20m,3.0,3.0,fixed,college,full_time,[git],[],[git],[frontend developer],[it phan mem],[frontend],"3 years of hands-on WordPress publishing/site build experience. Good HTML/CSS/JS; PHP for theme/template tweaks. WordPress admin skills: Pages/Posts, Menus, Media, Users, Permalinks. Confident wit...","Build websites and landing pages on WordPress Customize and extend themes/plugins as required by each project. Publish content: create/edit Pages Posts, set categories/tags, featured image, slug/e...","Competitive salary, negotiable based on capability. 85% salary during probation, with annual salary review. 13th-month salary, holiday bonuses, and Best Member awards. Training and development pro...",Vị trí tuyển dụng: WordPress Developer - Senior Level wordpress developer - level\n\nNhóm công việc: software_engineering\n\nCấp độ công việc: senior\n\nLoại hình tuyển dụng: full_time\n\nYêu cầu ...,1,0
2,https://www.topcv.vn/brand/concentrixservices/tuyen-dung/data-processing-analyst-j2102076.html,None,Data Processing Analyst,data labeling specialist,data_governance_quality,junior,TP. Hồ Chí Minh,"Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận Hồ Chí Minh",False,unknown,25000000.0,30000000.0,False,30m_50m,2.0,2.0,fixed,bachelor,full_time,"[python, sql, git, r, communication, problem solving, api]","[python, sql, git, r, communication, problem solving, api]",[],[data analyst],[du lieu],[data],"The Must-Haves Strong understanding of data analytics, data processing and scripting. Experience working with Python, R, MySQL, and other SQL languages . Proficiency with Git a plus Communication,...","Get to know the role We are looking for a new Analyst for one of our Data Processing Teams. We believe a successful candidate has problem solving and python programming skills, but if you believe ...","2 months probation full salary Join the insurance regimes according 

(1477, 31)


In [134]:
section_cols = [
    "section_embedding_row_id",
    "job_url", "job_title_display", "job_title_canonical",
    "job_family", "job_level_norm", "location_city", "work_mode",
    "section_type", "chunk_order", "section_priority",
    "chunk_text_raw", "chunk_text_phobert",
    "has_dense_embedding"
]

section_cols = [c for c in section_cols if c in job_sections_df.columns]

job_sections_ready = job_sections_df[section_cols].copy()
display(job_sections_ready.head(3))
print(job_sections_ready.shape)

,job_url,job_title_display,job_title_canonical,job_family,job_level_norm,location_city,work_mode,section_type,chunk_order,section_priority,chunk_text_raw,chunk_text_phobert,has_dense_embedding
0,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,title,0,5.0,Nhân Viên Tester Mobile App automation tester,Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Tiêu đề\nNhân Viên Tester ...,0
1,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,requirements,0,4.5,"Tốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan. Có kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói riêng. Có kinh nghiệm về sản phẩm Product android Nắ...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nTốt nghiệp chuyên...,0
2,https://www.topcv.vn/brand/ahv/tuyen-dung/nhan-vien-tester-mobile-app-android-di-lam-ngay-j1947367.html,Nhân Viên Tester Mobile App,automation tester,qa_testing,junior,TP. Hà Nội,onsite,requirements,1,4.5,"est mobile game hoặc app có lượng người dùng lớn\nTốt nghiệp chuyên ngành CNTT, Khoa học máy tính hoặc lĩnh vực liên quan.\nCó kiến thức tốt về kiểm thử phần mềm nói chung và kiểm thử mobile nói r...",Vị trí: Nhân Viên Tester Mobile App automation tester\nNhóm nghề: qa_testing\nCấp độ: junior\nĐịa điểm: TP. Hà Nội\nHình thức làm việc: onsite\nKỹ năng chính: api\nPhần: Yêu cầu\nest mobile game h...,0


(11677, 13)


In [136]:
artifact_paths = {}

if SAVE_INTERMEDIATE:
    artifact_paths["jobs_matching_ready_v3"] = save_table(
        df_matching_ready, ARTIFACT_DIR / "jobs_matching_ready_v3"
    )

    artifact_paths["jobs_chatbot_ready_v3"] = save_table(
        df_chatbot_ready, ARTIFACT_DIR / "jobs_chatbot_ready_v3"
    )

    artifact_paths["jobs_chatbot_sections_v3"] = save_table(
        job_sections_ready, ARTIFACT_DIR / "jobs_chatbot_sections_v3"
    )

    artifact_paths["job_skill_map_v3"] = save_table(
        job_skill_map_df, ARTIFACT_DIR / "job_skill_map_v3"
    )

    artifact_paths["role_skill_stats_v3"] = save_table(
        role_skill_stats_df, ARTIFACT_DIR / "role_skill_stats_v3"
    )

    artifact_paths["job_embedding_index_v3"] = save_table(
        df_clean[
            [
                "job_embedding_row_id",
                "job_url",
                "job_title_display",
                "job_title_canonical",
                "job_family",
                "job_level_norm",
                "location_city",
                "work_mode",
                "job_text_phobert_match",
            ]
        ],
        ARTIFACT_DIR / "job_embedding_index_v3"
    )

    if len(job_sections_df) > 0:
        artifact_paths["job_section_embedding_index_v3"] = save_table(
            job_sections_df[
                [
                    "section_embedding_row_id",
                    "job_url",
                    "job_title_display",
                    "job_title_canonical",
                    "job_family",
                    "job_level_norm",
                    "location_city",
                    "section_type",
                    "chunk_order",
                    "chunk_text_raw",
                    "chunk_text_phobert",
                ]
            ],
            ARTIFACT_DIR / "job_section_embedding_index_v3"
        )

    if RUN_EMBEDDING and job_dense_embeddings is not None:
        job_emb_path = ARTIFACT_DIR / "job_dense_embeddings_v3.npy"
        np.save(job_emb_path, job_dense_embeddings)
        artifact_paths["job_dense_embeddings_v3"] = str(job_emb_path)

    if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and section_dense_embeddings is not None:
        section_emb_path = ARTIFACT_DIR / "section_dense_embeddings_v3.npy"
        np.save(section_emb_path, section_dense_embeddings)
        artifact_paths["section_dense_embeddings_v3"] = str(section_emb_path)

    print("[INFO] Saved main artifacts.")
    display(pd.Series(artifact_paths))
else:
    print("[INFO] SAVE_INTERMEDIATE=False -> skip writing artifacts.")

[INFO] SAVE_INTERMEDIATE=False -> skip writing artifacts.


In [137]:
manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "run_timestamp": datetime.now().isoformat(),
    "raw_input_path": str(RAW_INPUT_PATH),
    "artifact_dir": str(ARTIFACT_DIR.resolve()),
    "n_jobs": int(len(df_clean)),
    "n_sections": int(len(job_sections_df)),
    "embedding_config": {
        "model_name": PHOBERT_MODEL_NAME,
        "metric": "cosine",
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "job_text_field": "job_text_phobert_match",
        "job_text_prepared_field": "job_text_phobert_match_prepared" if "job_text_phobert_match_prepared" in df_clean.columns else None,
        "chatbot_text_field": "job_text_phobert_chatbot",
        "chatbot_text_prepared_field": "job_text_phobert_chatbot_prepared" if "job_text_phobert_chatbot_prepared" in df_clean.columns else None,
        "section_text_field": "chunk_text_phobert",
        "section_text_prepared_field": "chunk_text_phobert_prepared" if "chunk_text_phobert_prepared" in job_sections_df.columns else None,
        "job_embedding_id_field": "job_embedding_row_id",
        "section_embedding_id_field": "section_embedding_row_id",
        "max_length_match": PHOBERT_MAX_LENGTH_MATCH,
        "max_length_chatbot": PHOBERT_MAX_LENGTH_CHATBOT,
        "max_length_chunk": PHOBERT_MAX_LENGTH_CHUNK,
        "batch_size": PHOBERT_BATCH_SIZE,
        "segmentation": "underthesea_if_available",
    },
    "core_fields": {
        "title_display": "job_title_display",
        "title_canonical": "job_title_canonical",
        "job_family": "job_family",
        "job_level": "job_level_norm",
        "location": "location_city",
        "working_address": "working_address_clean",
        "work_mode": "work_mode",
        "salary_min": "salary_min_vnd_month",
        "salary_max": "salary_max_vnd_month",
        "experience_min": "experience_min_years",
        "experience_max": "experience_max_years",
        "education_level": "education_level_norm",
        "employment_type": "employment_type_norm",
        "skills": "skills_extracted",
        "skills_required": "skills_required",
        "skills_preferred": "skills_preferred",
        "tags_model_text": "tags_text_for_model",
    },
    "downstream_field_guide": DOWNSTREAM_FIELD_GUIDE,
    "artifacts": artifact_paths,
}

if SAVE_INTERMEDIATE:
    manifest_path = ARTIFACT_DIR / "manifest_v3_phobert.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("[INFO] Manifest saved:", manifest_path)
else:
    print("[INFO] SAVE_INTERMEDIATE=False -> skip manifest write.")

display(pd.Series(manifest))

[INFO] SAVE_INTERMEDIATE=False -> skip manifest write.


notebook_version                                                                                                                                                                                         preprocessing_v3_phobert
run_timestamp                                                                                                                                                                                          2026-04-02T23:13:49.227805
raw_input_path                                                                                                                                             ..\..\..\data\raw\jobs\topcv_all_fields_merged_2026-04-01_19-41-42.csv
artifact_dir                                                                                                                                  D:\TTTN\job_recommendation_system\src\data_processing\outputs_preprocessing_v3_test
n_jobs                                                                                          

In [138]:
length_stats = pd.DataFrame({
    "job_text_sparse_len": df_clean["job_text_sparse"].fillna("").str.len(),
    "job_text_phobert_match_len": df_clean["job_text_phobert_match"].fillna("").str.len(),
    "job_text_phobert_chatbot_len": df_clean["job_text_phobert_chatbot"].fillna("").str.len(),
    "tags_text_for_model_len": df_clean["tags_text_for_model"].fillna("").str.len() if "tags_text_for_model" in df_clean.columns else 0,
    "skills_text_phobert_len": df_clean["skills_text_phobert"].fillna("").str.len() if "skills_text_phobert" in df_clean.columns else 0,
    "skills_required_text_phobert_len": df_clean["skills_required_text_phobert"].fillna("").str.len() if "skills_required_text_phobert" in df_clean.columns else 0,
})

display(length_stats.describe())

if len(job_sections_df) > 0:
    section_stats = (
        job_sections_df.assign(
            chunk_len=job_sections_df["chunk_text_phobert"].fillna("").str.len()
        )
        .groupby("section_type")["chunk_len"]
        .describe()
        .reset_index()
    )
    display(section_stats)

print("Avg skills per job:", df_clean["skills_extracted"].apply(len).mean())
print("Jobs without extracted skills:", (df_clean["skills_extracted"].apply(len) == 0).sum())
print("Avg required skills per job:", df_clean["skills_required"].apply(len).mean())
print("Avg preferred skills per job:", df_clean["skills_preferred"].apply(len).mean())

,job_text_sparse_len,job_text_phobert_match_len,job_text_phobert_chatbot_len,tags_text_for_model_len,skills_text_phobert_len,skills_required_text_phobert_len
count,1477.000000,1477.000000,1477.000000,1477.000000,1477.000000,1477.000000
mean,1872.713609,990.723087,2495.488152,44.356127,30.794854,25.161137
std,1063.133449,239.677391,624.622581,21.805041,26.685497,24.129782
min,32.000000,124.000000,167.000000,0.000000,0.000000,0.000000
25%,1190.000000,813.000000,2041.000000,30.000000,13.000000,8.000000
50%,1611.000000,1011.000000,2468.000000,38.000000,23.000000,18.000000
75%,2165.000000,1141.000000,2913.000000,58.000000,44.000000,35.000000
max,10694.000000,1798.000000,4939.000000,139.000000,159.000000,148.000000


,section_type,count,mean,std,min,25%,50%,75%,max
0,benefits,2313.0,647.992218,200.402020,181.0,470.00,691.0,833.0,939.0
1,company,2681.0,679.741887,211.819949,143.0,520.00,798.0,850.0,931.0
2,description,2708.0,729.736337,203.943267,200.0,563.75,825.0,892.0,1285.0
3,requirements,2498.0,715.032426,205.964938,216.0,550.00,792.5,887.0,1238.0
4,title,1477.0,228.820582,50.427908,92.0,195.00,224.0,254.0,637.0


Avg skills per job: 3.5368991198375084
Jobs without extracted skills: 125
Avg required skills per job: 2.9322951929587
Avg preferred skills per job: 0.17670954637779282


In [139]:
# =========================
# FINAL QA SUMMARY
# =========================

def pct_missing(series):
    return float(series.isna().mean()) if len(series) > 0 else 0.0

def pct_unknown(series):
    s = series.fillna("").astype(str).str.strip().str.lower()
    return float((s == "unknown").mean()) if len(series) > 0 else 0.0

qa_summary = {
    "n_jobs": int(len(df_clean)),
    "n_sections": int(len(job_sections_df)),

    # Title
    "pct_missing_job_title_display": pct_missing(df_clean["job_title_display"]) if "job_title_display" in df_clean.columns else None,
    "pct_missing_job_title_canonical": pct_missing(df_clean["job_title_canonical"]) if "job_title_canonical" in df_clean.columns else None,
    "pct_unknown_job_family": pct_unknown(df_clean["job_family"]) if "job_family" in df_clean.columns else None,

    # Location
    "pct_missing_location_city": pct_missing(df_clean["location_city"]) if "location_city" in df_clean.columns else None,
    "pct_true_is_multi_location": float(df_clean["is_multi_location"].fillna(False).mean()) if "is_multi_location" in df_clean.columns else None,

    # Work mode
    "pct_unknown_work_mode": pct_unknown(df_clean["work_mode"]) if "work_mode" in df_clean.columns else None,

    # Salary
    "pct_missing_salary_min": pct_missing(df_clean["salary_min_vnd_month"]) if "salary_min_vnd_month" in df_clean.columns else None,
    "pct_missing_salary_max": pct_missing(df_clean["salary_max_vnd_month"]) if "salary_max_vnd_month" in df_clean.columns else None,
    "pct_negotiable_salary": float(df_clean["salary_is_negotiable"].fillna(False).mean()) if "salary_is_negotiable" in df_clean.columns else None,

    # Experience
    "pct_missing_experience_min": pct_missing(df_clean["experience_min_years"]) if "experience_min_years" in df_clean.columns else None,
    "pct_missing_experience_max": pct_missing(df_clean["experience_max_years"]) if "experience_max_years" in df_clean.columns else None,
    "pct_unknown_experience_type": pct_unknown(df_clean["experience_type"]) if "experience_type" in df_clean.columns else None,

    # Education / employment / level
    "pct_unknown_education_level": pct_unknown(df_clean["education_level_norm"]) if "education_level_norm" in df_clean.columns else None,
    "pct_unknown_employment_type": pct_unknown(df_clean["employment_type_norm"]) if "employment_type_norm" in df_clean.columns else None,
    "pct_unknown_job_level": pct_unknown(df_clean["job_level_norm"]) if "job_level_norm" in df_clean.columns else None,

    # Tags / skills
    "avg_tags_per_job": float(df_clean["tags_list"].apply(len).mean()) if "tags_list" in df_clean.columns else None,
    "avg_skills_per_job": float(df_clean["skills_extracted"].apply(len).mean()) if "skills_extracted" in df_clean.columns else None,
    "avg_required_skills_per_job": float(df_clean["skills_required"].apply(len).mean()) if "skills_required" in df_clean.columns else None,
    "avg_preferred_skills_per_job": float(df_clean["skills_preferred"].apply(len).mean()) if "skills_preferred" in df_clean.columns else None,
    "pct_jobs_without_skills": float((df_clean["skills_extracted"].apply(len) == 0).mean()) if "skills_extracted" in df_clean.columns else None,

    # Text lengths
    "avg_job_text_sparse_len": float(df_clean["job_text_sparse"].fillna("").str.len().mean()) if "job_text_sparse" in df_clean.columns else None,
    "avg_job_text_match_len": float(df_clean["job_text_phobert_match"].fillna("").str.len().mean()) if "job_text_phobert_match" in df_clean.columns else None,
    "avg_job_text_chatbot_len": float(df_clean["job_text_phobert_chatbot"].fillna("").str.len().mean()) if "job_text_phobert_chatbot" in df_clean.columns else None,
    "avg_section_chunk_len": float(job_sections_df["chunk_text_phobert"].fillna("").str.len().mean()) if "chunk_text_phobert" in job_sections_df.columns and len(job_sections_df) > 0 else None,

    # Embedding readiness
    "job_has_dense_embedding_rate": float(df_clean["has_dense_embedding"].fillna(0).mean()) if "has_dense_embedding" in df_clean.columns else None,
    "section_has_dense_embedding_rate": float(job_sections_df["has_dense_embedding"].fillna(0).mean()) if "has_dense_embedding" in job_sections_df.columns and len(job_sections_df) > 0 else None,
}

qa_summary_series = pd.Series(qa_summary)
display(qa_summary_series)

print("\n=== DISTRIBUTIONS ===")
if "job_family" in df_clean.columns:
    print("\njob_family:")
    display(df_clean["job_family"].value_counts(dropna=False).head(15))

if "work_mode" in df_clean.columns:
    print("\nwork_mode:")
    display(df_clean["work_mode"].value_counts(dropna=False))

if "salary_bucket" in df_clean.columns:
    print("\nsalary_bucket:")
    display(df_clean["salary_bucket"].value_counts(dropna=False))

if "experience_type" in df_clean.columns:
    print("\nexperience_type:")
    display(df_clean["experience_type"].value_counts(dropna=False))

if "education_level_norm" in df_clean.columns:
    print("\neducation_level_norm:")
    display(df_clean["education_level_norm"].value_counts(dropna=False))

if "employment_type_norm" in df_clean.columns:
    print("\nemployment_type_norm:")
    display(df_clean["employment_type_norm"].value_counts(dropna=False))

if "job_level_norm" in df_clean.columns:
    print("\njob_level_norm:")
    display(df_clean["job_level_norm"].value_counts(dropna=False))

print("\n=== SANITY CHECK SAMPLES ===")

# sample 1: jobs without skills
if "skills_extracted" in df_clean.columns:
    display(df_clean.loc[
        df_clean["skills_extracted"].apply(len) == 0,
        [
            c for c in [
                "job_title_raw", "job_title_display", "job_family",
                "tags_raw", "requirements_raw", "description_raw"
            ] if c in df_clean.columns
        ]
    ].head(5))

# sample 2: unknown work mode
if "work_mode" in df_clean.columns:
    display(df_clean.loc[
        df_clean["work_mode"].fillna("").eq("unknown"),
        [
            c for c in [
                "job_title_raw", "working_addresses_raw", "description_raw", "benefits_raw", "requirements_raw"
            ] if c in df_clean.columns
        ]
    ].head(5))

# sample 3: unknown job level
if "job_level_norm" in df_clean.columns:
    display(df_clean.loc[
        df_clean["job_level_norm"].fillna("").eq("unknown"),
        [
            c for c in [
                "job_title_raw", "job_level_raw", "experience_raw", "experience_min_years"
            ] if c in df_clean.columns
        ]
    ].head(5))

n_jobs                               1477.000000
n_sections                          11677.000000
pct_missing_job_title_display           0.000000
pct_missing_job_title_canonical         0.000000
pct_unknown_job_family                  0.075152
pct_missing_location_city               0.004739
pct_true_is_multi_location              0.043331
pct_unknown_work_mode                   0.692620
pct_missing_salary_min                  0.486798
pct_missing_salary_max                  0.486798
pct_negotiable_salary                   0.486798
pct_missing_experience_min              0.000000
pct_missing_experience_max              0.025728
pct_unknown_experience_type             0.000000
pct_unknown_education_level             0.002031
pct_unknown_employment_type             0.004062
pct_unknown_job_level                   0.000000
avg_tags_per_job                        2.260664
avg_skills_per_job                      3.536899
avg_required_skills_per_job             2.932295
avg_preferred_skills


=== DISTRIBUTIONS ===

job_family:


job_family
software_engineering       477
product_project_ba         319
data_analytics             143
data_science_ml            117
qa_testing                 111
unknown                    111
cloud_devops_sre            98
data_engineering            68
data_governance_quality     29
database_platform            4
Name: count, dtype: int64


work_mode:


work_mode
unknown    1023
hybrid      206
onsite      190
remote       58
Name: count, dtype: int64

...

job_level_norm
junior      612
middle      306
senior      211
fresher     102
lead         97
intern       90
manager      53
director      6
Name: count, dtype: int64


=== SANITY CHECK SAMPLES ===


,job_title_raw,job_title_display,job_family,tags_raw,requirements_raw,description_raw
20,Devops Engineer,DevOps Engineer,cloud_devops_sre,Chuyên môn Fullstack Developer,"Tốt nghiệp Đại học trở lên chuyên ngành, Marketing, Quản trị Kinh doanh, Công nghệ thông tin...; Tính chính xác trong công việc Ham học hỏi, trau dồi kiến thức Khả năng làm việc theo nhóm\nTốt ngh...","Khai các chiến lược giám sát, cảnh báo và ứng phó sự cố Monitoring DevOps: Prometheus, Grafana,... Triển khai và duy trì các biện pháp bảo mật tốt nhất trong tất cả các giai đoạn của vòng đời phát..."
25,Chuyên Viên Phân Tích Nghiệp Vụ ( Business Analyst ),Chuyên Viên Phân Tích Nghiệp Vụ (Business Analyst),product_project_ba,Chuyên môn Business Analyst (Phân tích nghiệp vụ); Y tế / Dược phẩm,"Tốt nghiệp CĐ/ĐH chuyên ngành CNTT hoặc Tài chính, chứng khoán, ngân hàng; Có kinh nghiệm từ 3 năm đảm nhiệm vị trí Business Analyst, ưu tiên có kinh nghiệm, hiểu biết về quy trình, nghiệp vụ liên...","Tiếp nhận, phân tích yêu cầu từ cán bộ tư vấn nghiệp vụ, hỗ trợ giải pháp cho các yêu cầu chức năng, báo cáo, tích hợp… Tham gia các dự án phát triển phần mềm với các khách hàng lớn của Công ty. T..."
36,Manual Tester Intern ( Thực Tập Sinh Kiểm Thử Phần Mềm ),Manual Tester Intern,qa_testing,Chuyên môn Software Tester (Automation & Manual); IT - Phần mềm,"Đang là sinh viên năm 4 hoặc đã tốt nghiệp CĐ/ĐH chuyên ngành CNTT hoặc tương đương; Yêu thích và có định hướng theo tester Có tính cẩn thận, tỉ mỉ, kiên nhẫn, có trách nhiệm trong công việc Ưu ti...","Hỗ trợ kiểm thử phần mềm cho các dự án. Hỗ trợ lập chiến lược test và kế hoạch thực hiện test. Được đào tạo về test plan, test design và test case. Lập báo cáo về tình trạng test. Phối hợp với các..."
39,Thực Tập Sinh Lập Trình ( Angular ),Thực Tập Sinh Lập Trình (Angular),software_engineering,Chuyên môn Frontend Developer,"Sinh viên năm cuối hoặc sắp tốt nghiệp các chuyên ngành liên quan đến Công nghệ thông tin, Khoa học máy tính hoặc các ngành tương đương; Nắm vững kiến thức cơ bản của framework Angular, hiểu về cấ...","Tham gia vào quá trình phát triển và xây dựng sản phẩm phần mềm. Hỗ trợ đội ngũ phát triển trong việc thiết kế, phát triển và kiểm thử các thành phần giao diện. Thực hiện bảo trì và sửa lỗi cho cá..."
41,Thực Tập Sinh Phân Tích Nghiệp Vụ ( Business Analyst Intern ),Thực Tập Sinh Phân Tích Nghiệp Vụ (Business Analyst Intern),product_project_ba,Chuyên môn Business Analyst (Phân tích nghiệp vụ); IT - Phần mềm,"Sinh viên năm 3, 4 hoặc mới tốt nghiệp CĐ/ĐH chuyên ngành CNTT. Có thể thực tập fulltime từ 3 tháng trở lên. Có kiến thức về phân tích thiết kế hệ thống hoặc công nghệ phần mềm là một lợi thế Tư d...","Tiếp nhận, phân tích và chủ động tư vấn các yêu cầu tính năng sản phẩm Lập và chuẩn bị các tài liệu, hồ sơ cho yêu cầu phát triển và chuyển cho bộ phận lập trình thực hiện. Tham gia phối hợp với t..."


,job_title_raw,working_addresses_raw,description_raw,benefits_raw,requirements_raw
2,Data Processing Analyst,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: CMC Creative Space, Phường Tân Thuận (Quận 7 cũ)\nHồ Chí Minh:","Get to know the role We are looking for a new Analyst for one of our Data Processing Teams. We believe a successful candidate has problem solving and python programming skills, but if you believe ...","2 months probation full salary Join the insurance regimes according to the provisions of the Labor Law. Accident insurance 24/7. 13th, 14th Monthly Salary 14 annual leaves day Tet, holiday's gifte...","The Must-Haves Strong understanding of data analytics, data processing and scripting. Experience working with Python, R, MySQL, and other SQL languages . Proficiency with Git a plus Communication,..."
6,Chuyên Viên Phân Tích Nghiệp Vụ ( Business Analyst ),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (quận Tha...",- Tiếp nhận yêu cầu trực tiếp từ các người dùng nội bộ trong công ty - Tiếp nhận yêu cầu từ PM/PO nội bộ team - Nghiên cứu và phân tích yêu cầu - Chủ động tổ chức họp và đánh giá sự ảnh hưởng của ...,"1. Thu nhập cạnh tranh: - Lương cứng: Thoả thuận theo năng lực - Cơ hội thăng tiến nhanh cùng dự án. - Sản phẩm hay, có tiềm năng phát triển mạnh trong lĩnh vực giáo dục trực tuyến - Được tặng bộ ...","- Có 1-2 năm kinh nghiệm phân tích yêu cầu và thiết kế giải pháp - Có kiến thức cơ bản về quy trình phát triển phần mềm - Biết viết các tài liệu yêu cầu bao gồm: SRS, FRS - Có khả năng phân tích d..."
7,Data Governance Specialist,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...","− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D..."
8,Middle Java Developer,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (quận Tha...","- Tham gia phát triển nền tảng dạy học trực tuyến. - Nâng cấp, bảo trì các hệ thống đã triển khai. - Xây dựng, thiết kế và cải tiến hệ thống theo chuẩn kiến trúc Microservices. - Tham gia đóng góp...","1. Thu nhập cạnh tranh: - Lương cứng: thoả thuận theo năng lực - Cơ hội thăng tiến nhanh cùng dự án. - Sản phẩm hay, có tiềm năng phát triển mạnh trong lĩnh vực giáo dục trực tuyến - Được tặng bộ ...","- Tối thiểu 2 năm kinh nghiệm làm các dự án sử dụng ngôn ngữ Java (Spring Framework) - Có kinh nghiệm lập trình với các cơ sở dữ liệu như: MySQL, Oracle, MongoDB, Redis... - Có hiểu biết về OOP, S..."
9,Nhân Viên Kiểm Thử Phần Mềm,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (quận Tha...","- Nghiên cứu, phân tích yêu cầu. - Test web và ứng dụng mobile. - Viết tài liệu cho sản phẩm. - Theo dõi, tổng hợp và báo cáo kế hoạch phát triển, test. - Thực hiện test và phối hợp với developer ...","Tinh thần: - Làm việc trong môi trường start-up có tốc độ tăng trưởng và phát triển nhanh. - Được chia sẻ cơ hội học tập chất lượng cao đến t

,job_title_raw,job_level_raw,experience_raw,experience_min_years


## Ghi chú sử dụng artifact

- `jobs_matching_ready_v3`: dùng cho matching / recommendation
  - text chính: `job_text_phobert_match`
  - metadata quan trọng:
    - `job_title_display`
    - `job_title_canonical`
    - `job_family`
    - `job_level_norm`
    - `location_city`
    - `work_mode`
    - `skills_extracted`
    - `skills_required`
    - `skills_preferred`
  - id embedding: `job_embedding_row_id`
  - vector tương ứng: `job_dense_embeddings_v3.npy` (nếu `RUN_EMBEDDING=True`)

- `jobs_chatbot_ready_v3`: dùng cho chatbot summary-level
  - text chính: `job_text_phobert_chatbot`
  - metadata quan trọng:
    - `location_city`
    - `working_address_clean`
    - `salary_min_vnd_month`
    - `salary_max_vnd_month`
    - `experience_min_years`
    - `experience_max_years`
    - `skills_required`
    - `skills_preferred`

- `jobs_chatbot_sections_v3`: dùng cho retrieval theo chunk / section
  - text chính: `chunk_text_phobert`
  - metadata quan trọng:
    - `section_type`
    - `chunk_order`
    - `section_priority`
    - `job_family`
    - `job_level_norm`
    - `location_city`
  - id embedding: `section_embedding_row_id`
  - vector tương ứng: `section_dense_embeddings_v3.npy` (nếu `RUN_EMBEDDING=True` và `RUN_SECTION_EMBEDDING=True`)

- `job_skill_map_v3`: bảng long-format job-skill
  - mỗi dòng là 1 skill extract được từ 1 job
  - có các trường:
    - `skill`
    - `skill_group`
    - `source_field`
    - `importance`
    - `excerpt`

- `role_skill_stats_v3`: thống kê skill theo nhóm nghề (`job_family`)

- `manifest_v3_phobert.json`: metadata toàn bộ pipeline
  - config embedding
  - downstream field guide
  - danh sách artifact đã lưu